# Shell & Tube Heat Exchanger — Bell-Delaware Design Suite

Interactive design notebook built from `bell_delaware_v3.py`, using the same user-input and fluid-property architecture as the *Plate Heat Exchanger* notebook.

| Cell | Content |
|---|---|
| 1 | Fluid property engine: water, ISO VG 46 oil, wort, 50 wt% ethanol, brine, any CoolProp pure fluid (`CP::Name`), and user-defined custom fluids |
| 2 | **Your Bell-Delaware engine, unchanged**: Tables 1-7, `Fluid`/`Config`, Ft, `evaluate()` (Steps 2-8), `design()` (Figure-1 loop), `sweep()` (Step 9), `report()` |
| 3 | Bridge: user inputs → `Fluid` objects (properties at mean bulk temperature, solves unknown outlet / flow, assumed-ε rule) |
| 4 | Verification suite: your original water-water benchmark, same case with the property engine, water-property check vs CoolProp, multi-fluid smoke test |
| 5 | Interactive dashboard (Design mode and Search mode) with pressure-drop and resistance plots |

**Run the cells in order (Runtime → Run all).** For CoolProp fluids run `!pip install CoolProp` first.

**Input conventions:** enter `0` for an unknown outlet temperature or flow rate. Give flow + both temperatures for one stream, and flow *or* outlet for the other. If both flows are given and both outlets are `0`, the assumed effectiveness ε is used.


In [ ]:
# ==============================================================================
# CELL 1: THERMOPHYSICAL PROPERTY ENGINE (water, industrial fluids, CoolProp, custom)
# ==============================================================================
# Same architecture as the Plate Heat Exchanger notebook:
#   1. Built-in calibrated models: water, ISO VG 46 oil, wort, 50 wt% ethanol, brine.
#   2. Dynamic CoolProp bridge (optional) for pure fluids - keys start with "CP::".
#   3. User-registered custom fluids (constant properties) from the dashboard.
#
# Differences from the PHE notebook (deliberate fixes):
#   * Water cp and k are now polynomial fits to IAPWS-95 (CoolProp), 1-150 degC,
#     max error < 0.05 %.  The earlier cp / k polynomials were up to 9 % / 4 % off.
#   * The CoolProp "Ethanol" entry no longer collides with the built-in 50 wt% mix
#     (built-in key is now "ethanol50").
#   * An unknown fluid name raises an error instead of silently returning water.
#   * CoolProp fluids are checked to be LIQUID at the given pressure (Bell-Delaware
#     as coded here is a single-phase liquid method).
# ==============================================================================

import math
import warnings
from typing import Tuple

try:
    import CoolProp.CoolProp as CP
    COOLPROP_AVAILABLE = True
except ImportError:
    CP = None
    COOLPROP_AVAILABLE = False
    # In Colab:  !pip install CoolProp   (then restart / re-run)

# Registered custom fluids: name -> (rho [kg/m3], cp [J/kgK], k [W/mK], mu [Pa s])
CUSTOM_FLUIDS = {}

BUILTIN_FLUIDS = ("water", "oil", "wort", "ethanol50", "brine")

# Polynomial fits (x = T/100, T in degC) to IAPWS-95 liquid water at 10 bar, 1-150 degC
_WATER_CP_COEF = [-99.1872961178, 462.009015673, -780.799677192, 684.696471667, -266.263599752, 4212.90020959]
_WATER_K_COEF = [0.0145453911672, -0.0686051978, 0.134773445647, -0.205009261923, 0.245407482262, 0.556648283424]


def _horner(coef, x):
    acc = 0.0
    for c in coef:
        acc = acc * x + c
    return acc


def water_properties(T_C: float) -> Tuple[float, float, float, float]:
    """
    Liquid water, 1-150 degC.
      rho : Kell (1975) formulation                         (< 0.15 % vs IAPWS-95)
      cp  : polynomial fit to IAPWS-95                      (< 0.05 %)
      k   : polynomial fit to IAPWS-95                      (< 0.05 %)
      mu  : Andrade-Vogel  mu = 2.414e-5 * 10^(247.8/(T-140))  (< 2 %)
    Returns (rho [kg/m3], cp [J/kgK], k [W/mK], mu [Pa s]).
    Temperatures outside 1-150 degC are clamped (with a warning).
    """
    T_in = float(T_C)
    if T_in < 1.0 or T_in > 150.0:
        warnings.warn(f"Water temperature {T_in:.1f} degC outside the 1-150 degC fit range; value clamped.")
    T = max(1.0, min(150.0, T_in))
    num = (999.83952 + 16.945176 * T - 7.9870401e-3 * T**2 - 46.170461e-6 * T**3
           + 105.56302e-9 * T**4 - 280.54253e-12 * T**5)
    rho = num / (1.0 + 16.897850e-3 * T)
    cp = _horner(_WATER_CP_COEF, T / 100.0)
    k = _horner(_WATER_K_COEF, T / 100.0)
    mu = 2.414e-5 * (10.0 ** (247.8 / (T + 273.15 - 140.0)))
    return float(rho), float(cp), float(k), float(mu)


def get_fluid_properties(name: str, T_C: float, P_bar: float = 2.0,
                         check_phase: bool = True) -> Tuple[float, float, float, float]:
    """
    Unified fluid property router.
      name    : "water" | "oil" | "wort" | "ethanol50" | "brine" | "CP::<CoolPropName>" | custom name
      T_C     : bulk temperature [degC]
      P_bar   : operating pressure [bar] (used by CoolProp fluids only)
      check_phase : CoolProp fluids must be liquid at (T, P); set False for wall-temperature lookups
    Returns (rho [kg/m3], cp [J/kgK], k [W/mK], mu [Pa s]).
    """
    clean = name.strip()

    # 1. user-defined custom fluids (constant properties)
    if clean in CUSTOM_FLUIDS:
        return CUSTOM_FLUIDS[clean]

    # 2. CoolProp pure fluids
    if clean.startswith("CP::"):
        if not COOLPROP_AVAILABLE:
            raise RuntimeError("CoolProp is not installed: run  pip install CoolProp  and re-run Cell 1.")
        fluid = clean[4:]
        T_K, P_Pa = float(T_C) + 273.15, float(P_bar) * 1e5
        try:
            if check_phase:
                phase = CP.PhaseSI("T", T_K, "P", P_Pa, fluid)
                if phase not in ("liquid", "supercritical_liquid"):
                    raise ValueError(f"{fluid} is '{phase}' at {T_C:.1f} degC and {P_bar:.2f} bar - "
                                     "this method needs a liquid; raise the stream pressure.")
            rho = CP.PropsSI("D", "T", T_K, "P", P_Pa, fluid)
            cp = CP.PropsSI("C", "T", T_K, "P", P_Pa, fluid)
            k = CP.PropsSI("L", "T", T_K, "P", P_Pa, fluid)
            mu = CP.PropsSI("V", "T", T_K, "P", P_Pa, fluid)
        except ValueError:
            raise
        except Exception as e:
            raise ValueError(f"CoolProp could not evaluate {fluid} at {T_C:.1f} degC / {P_bar:.2f} bar: {e}")
        return float(rho), float(cp), float(k), float(mu)

    # 3. built-in calibrated models
    key = clean.lower()
    if key == "water":
        return water_properties(T_C)
    if key == "oil":                       # ISO VG 46 industrial hydraulic oil
        T = max(10.0, min(120.0, float(T_C)))
        rho = 875.0 - 0.65 * (T - 15.0)
        cp = 1900.0 + 3.5 * T
        k = 0.132 - 0.00008 * T
        mu = 0.120 * math.exp(-0.042 * T)
        return rho, cp, k, mu
    if key == "wort":                      # 12 Plato brewer's wort
        rho_w, cp_w, k_w, mu_w = water_properties(T_C)
        return rho_w * 1.045, cp_w * 0.92, k_w * 0.94, mu_w * 1.55
    if key == "ethanol50":                 # 50 wt% ethanol-water
        T = max(10.0, min(90.0, float(T_C)))
        rho = 920.0 - 0.75 * T
        cp = 3600.0 + 4.2 * T
        k = 0.38 - 0.0005 * T
        mu = 0.0035 * math.exp(-0.022 * T)
        return rho, cp, k, mu
    if key == "brine":                     # geothermal / industrial saline brine
        rho_w, cp_w, k_w, mu_w = water_properties(T_C)
        return rho_w * 1.06, cp_w * 0.95, k_w * 0.97, mu_w * 1.15

    raise ValueError(f"Unknown fluid '{name}'. Use a built-in key {BUILTIN_FLUIDS}, 'CP::<name>' "
                     "or register a custom fluid.")


def fluid_menu():
    """(label, key) pairs for the dashboard dropdowns."""
    menu = [("Water (IAPWS fits)", "water"),
            ("ISO VG 46 Hydraulic Oil", "oil"),
            ("Brewer's Wort (12 Plato)", "wort"),
            ("50 wt% Ethanol-Water", "ethanol50"),
            ("Geothermal Saline Brine", "brine")]
    if COOLPROP_AVAILABLE:
        menu += [("Water (CoolProp IAPWS-95)", "CP::Water"),
                 ("Ethanol (CoolProp)", "CP::Ethanol"),
                 ("Methanol (CoolProp)", "CP::Methanol"),
                 ("Ethylene Glycol (CoolProp)", "CP::EthyleneGlycol"),
                 ("Acetone (CoolProp)", "CP::Acetone"),
                 ("Toluene (CoolProp)", "CP::Toluene")]
    return menu


print(f"Property engine ready. CoolProp available: {COOLPROP_AVAILABLE}")

Property engine ready. CoolProp available: False


In [ ]:
# ==============================================================================
# CELL 2: BELL-DELAWARE SHELL & TUBE CORE ENGINE  (your bell_delaware_v3.py, unchanged)
# ==============================================================================
# Everything below is your algorithm, copied verbatim: Tables 1-7, Fluid / Config,
# resolve_duty, LMTD, Ft, evaluate() (Steps 2-8), design() (Figure-1 loop),
# sweep() (Step 9) and report().  Only the hard-coded water benchmark / __main__
# block was removed (it now lives in Cell 4), and inputs come from Cells 3 and 5.
# ==============================================================================
"""
Bell-Delaware shell-and-tube heat exchanger design
==================================================
Implements the procedure in "Bell-Delaware Shell & Tube Design Procedure"
(based on ME6220 Lecture 12, IIT Madras).  Equation, table and figure numbers
in the comments refer to that document.

Flow of the code (mirrors Figure 1 of the document)
    Step 0  inputs, duty, properties           -> Fluid, Config, resolve_duty()
    Step 1  LMTD, R, P, Ft (+ shells in series)-> lmtd(), Ft_factor()
    Step 2  trial geometry from standard sizes -> evaluate()   (Eqs. 5-6, Tables 1-5)
    Step 3  tube side h_i, dP_t                -> evaluate()   (Eqs. 7-8)
    Step 4  shell-side geometry                -> evaluate()   (Eqs. 9-16)
    Step 5  ideal cross-flow h_oc              -> evaluate()   (Eqs. 17-19, Table 6)
    Step 6  correction factors Fn Fw Fb FL     -> evaluate()   (Eqs. 20-25)
    Step 7  overall coefficient U_o            -> evaluate()   (Eq. 26)
    Step 8  shell-side pressure drop           -> evaluate()   (Eqs. 27-33, Table 7)
    Step 9  iteration + checks                 -> design(), sweep()

Units: SI throughout (m, kg, s, W, K/degC, Pa).  Tube sizes in Config are in mm.

Things NOT in the document that this code had to assume (all flagged "ASSUMPTION"):
  * Table 4 constants are for Pt = 1.25 do; for other pitches the tube count is
    scaled by (1.25 do / Pt)^2.
  * Standard shell inside diameters (TEMA-style list, verify against your own table).
  * Nozzle sizing by rho*v^2 <= 2230 kg/(m s^2) when no nozzle diameter is given.
  * Shell-to-baffle clearance interpolated 2-5 mm with shell diameter.
  * Tube material is stainless steel only: wall conductivity WALL_K_STAINLESS = 16 W/mK
    (override with Config.kw) and the stainless wall thicknesses of Table 2.
  * Bundle-to-shell clearance is a function of bundle diameter, one function per head type
    (HEAD_CLEARANCE_M); the placeholders return constants until you fill them in.
  * Tubesheet thickness follows TEMA RCB-7.11 (expanded joints).  Corrosion allowance and the
    rounding step are inputs; the tubesheet is assumed to remove its thickness from the heat-transfer
    length (2 tubesheets, 1 for U-tube).  Switch off with Config.tubesheet_deducts_length.
  * Ds = Db + clearance(Db) is kept as computed (Config.round_shell_to_standard=False); set True to
    round up to the next standard shell instead.
  * Number and width of pass-partition lanes counted in Ab (Config.n_pass_lanes,
    Config.pass_lane_mm); the document says Ab includes them but gives no size.
"""
from __future__ import annotations

import itertools
import math
from dataclasses import dataclass, replace
from typing import Callable, Optional

import numpy as np

# =============================================================================
# TABLES (from the document)
# =============================================================================

# Table 1: typical overall coefficients (W/m2K) (low, high)
TABLE1_UO = {
    ("water", "water"): (800, 1500),
    ("steam", "water"): (1500, 4000),
    ("water", "organic solvents"): (250, 750),
    ("water", "light oils"): (350, 700),
    ("water", "heavy oils"): (60, 300),
    ("light oils", "light oils"): (100, 400),
    ("organic solvents", "organic solvents"): (100, 300),
    ("water", "gases"): (20, 300),
    ("gases", "gases"): (10, 50),
}

# Table 2 (stainless steel column only): OD (mm) -> wall thickness t in mm.
# The slides list 24.40 / 25.04 mm; the standard 1 in tube is 25.4 mm and is used here.
TUBES_MM = {
    15.88: 1.24,
    19.05: 1.65,
    25.40: 2.11,
    31.75: 2.77,
}

# Table 3: OD (mm) -> (clean min Pt, fouling min Pt) in mm
MIN_PITCH_MM = {
    15.88: (19.84, 22.22),
    19.05: (23.81, 25.40),
    25.40: (31.75, 31.75),
    31.75: (39.69, 39.69),
}

# Table 4: passes -> {'tri': (K1, n1), 'sq': (K1, n1)}   (Pt = 1.25 do)
BUNDLE_K = {
    1: {"tri": (0.319, 2.142), "sq": (0.215, 2.207)},
    2: {"tri": (0.249, 2.207), "sq": (0.156, 2.291)},
    4: {"tri": (0.175, 2.285), "sq": (0.158, 2.263)},
    6: {"tri": (0.0743, 2.499), "sq": (0.0402, 2.617)},
    8: {"tri": (0.0365, 2.675), "sq": (0.0331, 2.643)},
}

# Table 5: max baffle spacing (mm) for 19.05 mm steel tubes
MAX_BAFFLE_MM = {"segmental": 762.0, "double_segmental": 762.0, "full_circle": 1542.0}

# ASSUMPTION: standard shell inside diameters (inches -> m); verify with your TEMA table
STD_SHELLS_M = [d * 0.0254 for d in (8, 10, 12, 13.25, 15.25, 17.25, 19.25, 21.25, 23.25,
                                     25, 27, 29, 31, 33, 35, 37, 39, 42, 45, 48, 54, 60)]

# Step 2d: bundle-to-shell clearance (Ds - Db), m, as a FUNCTION OF BUNDLE DIAMETER Db (m).
# One function per head type; each takes Db in metres and returns the clearance in metres.
# Step 2d: bundle-to-shell clearance (Ds - Db), m, as a FUNCTION OF BUNDLE DIAMETER Db (m).
# One function per head type; each takes Db in metres and returns the clearance in metres.
def _clearance_fixed(Db: float) -> float:
    c1= 0.01*Db + 0.008
    return c1


def _clearance_u_tube(Db: float) -> float:
    c1= 0.01*Db + 0.008
    return c1


def _clearance_split_ring(Db: float) -> float:
    c1= 0.03*Db + 0.044
    return c1


def _clearance_pull_through(Db: float) -> float:
    c1= 0.008*Db + 0.0884
    return c1


HEAD_CLEARANCE_M = {
    "fixed": _clearance_fixed,
    "u_tube": _clearance_u_tube,
    "split_ring": _clearance_split_ring,
    "pull_through": _clearance_pull_through,
}


def head_clearance(head: str, Db: float) -> float:
    """Bundle-to-shell clearance (m) for a head type at bundle diameter Db (m)."""
    return HEAD_CLEARANCE_M[head](Db)


def bundle_dia_for_shell(Ds: float, head: str) -> float:
    """Db such that Db + clearance(Db) = Ds (Ds is known once a standard shell is chosen).
    Clearance depends on Db, so this is solved numerically by bisection.
    Assumes Db + clearance(Db) increases with Db (true for any sensible clearance curve)."""
    f = lambda Db: Db + head_clearance(head, Db) - Ds
    lo, hi = 0.0, Ds
    if f(lo) > 0 or f(hi) < 0:
        raise ValueError(f"Cannot find Db for Ds = {Ds:.4f} m with head '{head}': "
                         "check the clearance function (Db + clearance(Db) must reach Ds)")
    for _ in range(80):
        mid = 0.5 * (lo + hi)
        if f(mid) > 0:
            hi = mid
        else:
            lo = mid
    return 0.5 * (lo + hi)

# Tube material is fixed to stainless steel.
# ASSUMPTION: wall conductivity (W/mK) of stainless steel; override with Config.kw
WALL_K_STAINLESS = 16.0

# TEMA RCB-7.11: minimum tubesheet thickness with expanded tube joints.
# Tubesheets per bundle: tube length is lost to them (fixed / floating = 2, U-tube = 1).
TUBESHEETS_PER_HEAD = {"fixed": 2, "u_tube": 1, "split_ring": 2, "pull_through": 2}
TEMA_MIN_TOTAL_TS_MM = 19.1          # 3/4 in total thickness floor (R-7.11 and B-7.11, not C-7.11)


def min_tubesheet_mm(do_mm: float, tema_class: str, ca_mm: float = 0.0, round_mm: float = 1.0):
    """TEMA RCB-7.11 minimum TOTAL tubesheet thickness (mm), expanded tube joints.
    Returns (t_total_min, t_net_min) where t_net_min = total minus corrosion allowance.
      R : net >= tube OD;                       total >= 19.1 mm
      C : net >= 3/4 do (do <= 25.4), 22.2 (do <= 31.8), 25.4 (do <= 38.1), 31.8 (do <= 50.8)
      B : same net rule as C;                   total >= 19.1 mm
    The total is rounded UP to a multiple of round_mm (ASSUMPTION; 0 = no rounding)."""
    cls = tema_class.upper()
    if cls not in ("R", "C", "B"):
        raise ValueError("tema_class must be 'R', 'C' or 'B'")
    tol = 0.1                                                    # 31.75 mm tube counts as 1 1/4 in
    if cls == "R":
        net = do_mm
    elif do_mm <= 25.4 + tol:
        net = 0.75 * do_mm
    elif do_mm <= 31.8 + tol:
        net = 22.2
    elif do_mm <= 38.1 + tol:
        net = 25.4
    elif do_mm <= 50.8 + tol:
        net = 31.8
    else:
        raise ValueError(f"Tube OD {do_mm} mm is outside the TEMA RCB-7.11 table (max 50.8 mm)")
    total = net + ca_mm
    if cls in ("R", "B"):
        total = max(total, TEMA_MIN_TOTAL_TS_MM)
    if round_mm and round_mm > 0:
        total = math.ceil(total / round_mm - 1e-9) * round_mm
    return total, net


# Table 6: shell-side jh constants. rows by Re band: [1e4-1e5, 1e3-1e4, 1e2-1e3, 10-1e2, <10]
# each row = (a1, a2, a3, a4)
JH_TABLE = {
    30: [(0.321, -0.388, 1.45, 0.519), (0.321, -0.388, 1.45, 0.519), (0.593, -0.477, 1.45, 0.519),
         (1.36, -0.657, 1.45, 0.519), (1.4, -0.667, 1.45, 0.519)],
    45: [(0.37, -0.396, 1.93, 0.5), (0.37, -0.396, 1.93, 0.5), (0.73, -0.5, 1.93, 0.5),
         (1.498, -0.656, 1.93, 0.5), (1.55, -0.667, 1.93, 0.5)],
    90: [(0.37, -0.395, 1.187, 0.37), (0.107, -0.266, 1.187, 0.37), (0.408, -0.46, 1.187, 0.37),
         (0.9, -0.631, 1.187, 0.37), (0.97, -0.667, 1.187, 0.37)],
}
# Table 7: shell-side jf constants (b1, b2, b3, b4), same band order
JF_TABLE = {
    30: [(0.372, -0.123, 7.0, 0.5), (0.486, -0.152, 7.0, 0.5), (4.57, -0.476, 7.0, 0.5),
         (45.1, -0.973, 7.0, 0.5), (48.0, -1.0, 7.0, 0.5)],
    45: [(0.303, -0.126, 6.59, 0.52), (0.333, -0.136, 6.59, 0.52), (3.5, -0.476, 6.59, 0.52),
         (26.2, -0.913, 6.59, 0.52), (32.0, -1.0, 6.59, 0.52)],
    90: [(0.391, -0.148, 6.3, 0.378), (0.0815, 0.022, 6.3, 0.378), (6.09, -0.602, 6.3, 0.378),
         (32.1, -0.963, 6.3, 0.378), (35.0, -1.0, 6.3, 0.378)],
}

# Digitized charts (document says: by eye, about +/-0.02 -> guidance only)
FN_FIG9 = ([5, 10, 15, 20, 25, 30, 35], [0.955, 0.985, 1.015, 1.035, 1.047, 1.055, 1.060])
FW_FIG10 = ([0.0, 0.01, 0.03, 0.06, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.85, 0.88],
            [1.0, 1.07, 1.12, 1.145, 1.155, 1.152, 1.138, 1.07, 1.00, 0.93, 0.862, 0.795, 0.725, 0.68, 0.64])
BETA_L_FIG13 = ([0, 0.02, 0.05, 0.10, 0.20, 0.40, 0.60, 0.75], [0.03, 0.10, 0.14, 0.17, 0.21, 0.29, 0.36, 0.43])
BETA_LP_FIG13 = ([0, 0.02, 0.05, 0.10, 0.20, 0.40, 0.60, 0.78], [0.0, 0.10, 0.20, 0.30, 0.38, 0.48, 0.58, 0.68])


# =============================================================================
# DATA CLASSES
# =============================================================================
@dataclass
class Fluid:
    """One stream.  Leave m or T_out as None for ONE of the two streams and it is solved from the duty."""
    name: str
    T_in: float                      # degC
    T_out: Optional[float]           # degC
    m: Optional[float]               # kg/s
    rho: float                       # kg/m3  (at mean temperature)
    mu: float                        # Pa s   (at mean temperature)
    cp: float                        # J/kgK
    k: float                         # W/mK
    Rf: float = 0.0                  # fouling resistance, m2K/W (Table 4.5 of course text)
    mu_w: Optional[float] = None     # Pa s at wall; None -> mu (ratio 1, first pass)
    mu_of_T: Optional[Callable[[float], float]] = None   # optional: enables wall-temperature iteration

    @property
    def Pr(self) -> float:
        return self.mu * self.cp / self.k

    @property
    def T_mean(self) -> float:
        return 0.5 * (self.T_in + self.T_out)


@dataclass
class Config:
    # --- allocation and basic choices (Step 0 / 2) ---
    hot_in_tubes: bool = True
    Uo_assumed: float = 1000.0            # W/m2K, first guess (Table 1)
    kw: Optional[float] = None            # override wall conductivity
    do_mm: float = 19.05                  # Table 2 OD
    layout: int = 30                      # 30, 45 or 90 degrees
    Pt_mm: Optional[float] = None         # None -> Table 3 minimum
    shell_fouls: Optional[bool] = None    # None -> True if shell fluid Rf > 0
    passes: int = 2                       # 1, 2, 4, 6, 8
    L: float = 4.88                       # tube length, m (3.6-11 m)
    head: str = "fixed"                   # fixed | u_tube | split_ring | pull_through
    # --- tubesheet (TEMA RCB-7.11, expanded joints) ---
    tema_class: str = "B"                 # R | C | B
    corrosion_allowance_mm: float = 0.0   # tubesheet corrosion allowance, mm (set from your service)
    tubesheet_mm: Optional[float] = None  # None -> TEMA minimum; a given value is checked against it
    tubesheet_round_mm: float = 1.0       # ASSUMPTION: minimum is rounded up to this step
    # --- shell diameter ---
    round_shell_to_standard: bool = False # False: keep Ds = Db + clearance(Db) as computed (default)
                                          # True : round Ds up to the next standard shell (STD_SHELLS_M)
    tubesheet_deducts_length: bool = True # tube length L is overall: effective heat-transfer length
                                          # = L - n_tubesheets * thickness
    n_shells: Optional[int] = None        # None -> smallest n with Ft >= 0.8
    max_shells: int = 6
    # --- baffles (Step 2e) ---
    baffle_type: str = "segmental"
    Bc: float = 0.25                      # baffle cut (fraction of Ds)
    lB_frac: float = 0.2                  # lB = max(lB_frac*Ds, 50.8 mm), capped by Table 5
    delta_tb: float = 0.8e-3              # diametral tube-baffle clearance, m
    delta_sb: Optional[float] = None      # diametral shell-baffle clearance, m (None -> 2-5 mm by Ds)
    n_seal_pairs: int = 0                 # sealing-strip pairs crossed (Ns')
    # --- pass-partition lanes (Eq. 15: "Ab also includes any pass-partition lanes") ---
    n_pass_lanes: Optional[int] = None    # lanes lying PARALLEL to the cross-flow stream.
    #   None -> 0 for 1 pass, else 1.  Only lanes parallel to the flow bypass the bundle,
    #   so the count depends on how the partition plates sit relative to the baffle cut:
    #   a 2-pass exchanger has 1 lane, and it counts only if it is parallel to the flow;
    #   4/6/8 passes usually give 1-2 parallel lanes.  Set explicitly from your layout drawing.
    pass_lane_mm: float = 19.05           # lane width, m*1e3 (typically the partition plate + 2 x 6-10 mm)
    # --- options for the chart-vs-equation choices ---
    Fw_method: str = "chart"              # chart (Fig 10) | taborek (Eq 22) - do not mix
    FL_method: str = "chart"              # chart (Eq 24 + Fig 13) | taborek (Eq 25)
    # --- nozzles ---
    nozzle_d_shell: Optional[float] = None
    nozzle_d_tube: Optional[float] = None
    nozzle_rho_v2: float = 2230.0         # ASSUMPTION used to size nozzles when diameters not given
    # --- acceptance criteria (Step 7 / 9) ---
    # The document states Step 7 twice and the two readings differ when Nt is a whole number or
    # Ds is rounded up to a standard size:  "Uo calculated exceeds Uo assumed by 10-25%"  vs  "A_provided against
    # Q/(Uo Ft dTlm)".  Both are computed; accept_criterion picks which one drives the loop.
    accept_criterion: str = "area"        # area (A_prov vs A at Uo_calc) | Uo (Uo_calc vs Uo_assumed)
    margin_lo: float = 0.10
    margin_hi: float = 0.25
    LDs_lo: float = 5.0                   # Step 2c guide
    LDs_hi: float = 10.0
    dP_shell_max: float = 70e3            # Pa  (0.35-0.7 bar typical for liquids)
    dP_tube_max: float = 70e3
    liquid_velocity_check: bool = True    # tube-side 1-2.5 m/s test counts toward feasibility
    wall_iterations: int = 3              # only used when Fluid.mu_of_T is given
    max_iter: int = 25


# =============================================================================
# STEP 0 / 1 : duty, LMTD, Ft
# =============================================================================
def resolve_duty(hot: Fluid, cold: Fluid):
    """Eq. 1.  Fills in the missing flow rate or outlet temperature.  Returns Q (W)."""
    notes = []
    hot_full = hot.m is not None and hot.T_out is not None
    cold_full = cold.m is not None and cold.T_out is not None
    if hot_full:
        Q = hot.m * hot.cp * (hot.T_in - hot.T_out)
        if cold_full:
            Qc = cold.m * cold.cp * (cold.T_out - cold.T_in)
            if abs(Qc - Q) > 0.02 * Q:
                notes.append(f"Energy balance mismatch: Q_hot={Q:.4g} W, Q_cold={Qc:.4g} W (hot side used)")
        elif cold.T_out is None and cold.m is not None:
            cold.T_out = cold.T_in + Q / (cold.m * cold.cp)
        elif cold.m is None and cold.T_out is not None:
            cold.m = Q / (cold.cp * (cold.T_out - cold.T_in))
    elif cold_full:
        Q = cold.m * cold.cp * (cold.T_out - cold.T_in)
        if hot.T_out is None and hot.m is not None:
            hot.T_out = hot.T_in - Q / (hot.m * hot.cp)
        elif hot.m is None and hot.T_out is not None:
            hot.m = Q / (hot.cp * (hot.T_in - hot.T_out))
    else:
        raise ValueError("Specify flow and both temperatures for at least one stream.")
    return Q, notes


def lmtd(hot: Fluid, cold: Fluid) -> float:
    """Eq. 2, counter-current LMTD."""
    d1 = hot.T_in - cold.T_out
    d2 = hot.T_out - cold.T_in
    if d1 <= 0 or d2 <= 0:
        raise ValueError("Temperature cross / zero approach: no feasible counter-current LMTD.")
    return d1 if abs(d1 - d2) < 1e-9 * max(d1, d2) else (d1 - d2) / math.log(d1 / d2)


def R_P(hot: Fluid, cold: Fluid):
    """Eq. 3."""
    R = (hot.T_in - hot.T_out) / (cold.T_out - cold.T_in)
    P = (cold.T_out - cold.T_in) / (hot.T_in - cold.T_in)
    return R, P


def _ft_single(R: float, P: float) -> float:
    """First four lines of Eq. 4 (one shell, even number of tube passes). 0 if infeasible."""
    if P <= 0:
        return 1.0
    if abs(R - 1.0) < 1e-9:
        num = P * math.sqrt(2) / (1 - P)
        arg = (2 - P * (2 - math.sqrt(2))) / (2 - P * (2 + math.sqrt(2)))
        if arg <= 1 or P >= 1:
            return 0.0
        return num / math.log(arg)
    q = math.sqrt(R * R + 1)
    a = (1 - P) / (1 - P * R)
    d1 = 2 - P * (R + 1 + q)
    d2 = 2 - P * (R + 1 - q)
    if a <= 0 or d1 <= 0 or d2 <= 0:
        return 0.0
    b = d2 / d1
    if b <= 0 or math.log(b) == 0:
        return 0.0
    return q / (R - 1) * math.log(a) / math.log(b)


def Ft_factor(R: float, P: float, n: int = 1) -> float:
    """Eq. 4 including n shells in series via P* (Hewitt 3.77/3.78)."""
    if n == 1:
        return _ft_single(R, P)
    if abs(R - 1.0) < 1e-9:
        Ps = P / (n - P * (n - 1))
    else:
        X = (P * R - 1) / (P - 1)
        if X <= 0:
            return 0.0
        Xn = X ** (1.0 / n)
        Ps = (1 - Xn) / (R - Xn)
    return _ft_single(R, Ps)


def choose_shells(R: float, P: float, cfg: Config):
    if cfg.n_shells:
        return cfg.n_shells, Ft_factor(R, P, cfg.n_shells)
    for n in range(1, cfg.max_shells + 1):
        ft = Ft_factor(R, P, n)
        if ft >= 0.8:
            return n, ft
    return cfg.max_shells, Ft_factor(R, P, cfg.max_shells)


# =============================================================================
# SMALL HELPERS
# =============================================================================
def _band(Re: float) -> int:
    """Row index into Tables 6/7."""
    if Re >= 1e4:
        return 0
    if Re >= 1e3:
        return 1
    if Re >= 1e2:
        return 2
    if Re >= 10:
        return 3
    return 4


def jh_shell(Re, layout, Pt_over_do):   # Eq. 18
    a1, a2, a3, a4 = JH_TABLE[layout][_band(Re)]
    a = a3 / (1 + 0.14 * Re ** a4)
    return a1 * (1.33 / Pt_over_do) ** a * Re ** a2


def jf_shell(Re, layout, Pt_over_do):   # Eq. 28
    b1, b2, b3, b4 = JF_TABLE[layout][_band(Re)]
    b = b3 / (1 + 0.14 * Re ** b4)
    return b1 * (1.33 / Pt_over_do) ** b * Re ** b2


def _layout_factors(layout, Pt):
    if layout == 30:
        return 0.866 * Pt, 0.5 * Pt
    if layout == 45:
        return 0.707 * Pt, 0.707 * Pt
    if layout == 90:
        return Pt, Pt
    raise ValueError("layout must be 30, 45 or 90")


def _nozzle_dp(m, rho, d, rho_v2):
    """About one velocity head in + 0.5 out (Step 8). d sized from rho*v^2 if not given."""
    A = m / math.sqrt(rho_v2 * rho) if d is None else math.pi * d * d / 4
    u = m / (rho * A)
    return 1.5 * rho * u * u / 2, math.sqrt(4 * A / math.pi), u


def _bypass_factor(alpha, Ab_As, ratio):  # Eq. 23 (also used for F'b with the larger alpha)
    if ratio >= 0.5:
        return 1.0
    return math.exp(-alpha * Ab_As * (1 - (2 * ratio) ** (1.0 / 3.0)))


def window_geometry(Bc):  # Eq. 12
    th = 2 * math.acos(1 - 2 * Bc)
    Ra = (th - math.sin(th)) / (2 * math.pi)
    return th, Ra


def crossflow_fraction(Ds, Bc, Dctl):  # Eq. 13
    x = max(-1.0, min(1.0, Ds * (1 - 2 * Bc) / Dctl))
    ac = math.acos(x)
    Fc = (math.pi + 2 * x * math.sin(ac) - 2 * ac) / math.pi
    return Fc, (1 - Fc) / 2


def next_standard_shell(D):
    for s in STD_SHELLS_M:
        if s >= D:
            return s
    return None


# =============================================================================
# STEPS 2-8 : evaluate one trial geometry for a given assumed Uo
# =============================================================================
def evaluate(cfg: Config, ctx: dict, Uo_assumed: float, force_Ds: Optional[float] = None) -> dict:
    W = []                                   # warnings
    tube, shell = ctx["tube"], ctx["shell"]
    n = ctx["n_shells"]
    Ft, dTlm, Q = ctx["Ft"], ctx["dTlm"], ctx["Q"]
    do = cfg.do_mm * 1e-3
    t = ctx["t"]
    di = do - 2 * t
    L = cfg.L                        # overall tube length (tube-side flow path, dP_t)
    Leff = ctx["L_eff"]              # heat-transfer length: L minus tubesheet thickness
    Pt = ctx["Pt"]
    layout = cfg.layout

    # ---------------- Step 2: area -> Nt -> Db -> Ds -> recount ----------------
    A_req_total = Q / (Uo_assumed * Ft * dTlm)                    # Eq. 5
    A_req_shell = A_req_total / n
    Nt0 = A_req_shell / (math.pi * do * Leff)                        # Eq. 5
    K1, n1 = BUNDLE_K[cfg.passes]["tri" if layout == 30 else "sq"]
    pf = (Pt / (1.25 * do)) ** 2                                  # ASSUMPTION: pitch scaling of Table 4
    Db0 = do * (Nt0 * pf / K1) ** (1.0 / n1)                      # Eq. 6
    if force_Ds is None and not cfg.round_shell_to_standard:
        # Ds = Db + clearance(Db), kept exactly as computed (no rounding to a standard shell size).
        # Nt is made a whole number first (rounded UP so the provided area is not below the
        # required area; even for U-tubes), then Db follows from Eq. 6 with that Nt.
        Nt = math.ceil(Nt0 - 1e-9)
        if cfg.head == "u_tube":
            Nt += Nt % 2
        Nt = max(Nt, cfg.passes)
        Db = do * (Nt * pf / K1) ** (1.0 / n1)                    # Eq. 6 with the actual Nt
        cl = head_clearance(cfg.head, Db)
        Ds = Db + cl
    else:
        cl0 = head_clearance(cfg.head, Db0)                       # clearance = f(Db), trial bundle
        Ds = force_Ds if force_Ds is not None else next_standard_shell(Db0 + cl0)
        if Ds is None:
            return {"ok": False, "reason": "Required shell exceeds largest standard size; use more shells / longer tubes / higher Uo",
                    "Db_trial": Db0 + cl0, "Uo_assumed": Uo_assumed, "warnings": W}
        Db = bundle_dia_for_shell(Ds, cfg.head)                   # Db + clearance(Db) = Ds
        cl = Ds - Db
        Nt = int(math.floor(K1 * (Db / do) ** n1 / pf))           # Step 2d recount
        if cfg.head == "u_tube":
            Nt -= Nt % 2
        Nt = max(Nt, cfg.passes)
    A_prov_shell = Nt * math.pi * do * Leff
    A_prov_total = n * A_prov_shell
    Uo_req = Q / (A_prov_total * Ft * dTlm)                       # Uo needed with the real area
    if not (cfg.LDs_lo <= L / Ds <= cfg.LDs_hi):
        W.append(f"L/Ds = {L / Ds:.1f} outside the {cfg.LDs_lo:.0f}-{cfg.LDs_hi:.0f} guide (Step 2c)")

    # ---------------- Step 2e: baffles ----------------
    lB_max = MAX_BAFFLE_MM[cfg.baffle_type] * 1e-3
    lB_target = min(max(cfg.lB_frac * Ds, 0.0508), lB_max)
    n_sp = max(2, math.ceil(Leff / lB_target - 1e-9))
    lB = Leff / n_sp                                                 # <= lB_target so TEMA max respected
    Nb = n_sp - 1                                                 # Nb = L/lB - 1
    if cfg.do_mm != 19.05:
        W.append("Table 5 max baffle spacing is for 19.05 mm steel tubes; check the TEMA table for your tube")
    delta_sb = cfg.delta_sb if cfg.delta_sb is not None else (2.0 + 3.0 * min(max((Ds - 0.3) / 1.2, 0), 1)) * 1e-3
    delta_tb = cfg.delta_tb

    # ---------------- Step 4: shell-side geometry ----------------
    Ppl, Pt_eff = _layout_factors(layout, Pt)
    Dotl = Db - do                                                # Eq. 9
    As = lB * ((Ds - Dotl) + (Dotl - do) * (Pt - do) / Pt_eff)    # Eq. 10
    Ncv = Ds * (1 - 2 * cfg.Bc) / Ppl                             # Eq. 11
    theta_b, Ra = window_geometry(cfg.Bc)                         # Eq. 12
    Aw = Ra * math.pi * Ds ** 2 / 4
    Fc, Rw = crossflow_fraction(Ds, cfg.Bc, Dotl)                 # Eq. 13 (Dctl = Dotl)
    Nw = Rw * Nt                                                  # Eq. 14
    Awt = Nw * math.pi * do ** 2 / 4
    Aw_net = Aw - Awt
    if Aw_net <= 0:
        return {"ok": False, "reason": f"Window fully blocked by tubes (Aw = {Aw:.4f} m2, tubes {Awt:.4f} m2): "
                                       "increase the baffle cut or reduce Nt",
                "Uo_assumed": Uo_assumed, "warnings": W}
    # Eq. 15.  Ab = shell-to-bundle clearance + any pass-partition lanes parallel to the flow.
    n_lanes = cfg.n_pass_lanes if cfg.n_pass_lanes is not None else (0 if cfg.passes == 1 else 1)
    Ab_lanes = n_lanes * cfg.pass_lane_mm * 1e-3 * lB
    Ab = lB * (Ds - Db) + Ab_lanes
    Atb = math.pi * do * delta_tb * Nt * (1 + Fc) / 4             # Eq. 16
    Asb = Ds * delta_sb / 2 * (math.pi - theta_b / 2)
    AL = Atb + Asb
    Ab_As, AL_As = Ab / As, AL / As
    if Ab_As > 0.2:
        W.append(f"Ab/As = {Ab_As:.2f} > 0.2: add sealing strips or reduce Ds-Db (Step 9)")

    # ---------------- Shell-side flow and ideal bank (Step 5) ----------------
    Ws = shell.m
    Gs = Ws / As                                                  # Eq. 17
    Re_s = Gs * do / shell.mu
    us = Gs / shell.rho
    jh = jh_shell(Re_s, layout, Pt / do)                          # Eq. 18

    # ---------------- Tube side geometry (Step 3) ----------------
    Np = cfg.passes
    At_flow = Nt / Np * math.pi * di ** 2 / 4
    ut = tube.m / (tube.rho * At_flow)
    Re_t = tube.rho * ut * di / tube.mu

    # ---------------- Step 6 factors that do not depend on mu_w ----------------
    ratio_strips = cfg.n_seal_pairs / Ncv
    alpha = 1.5 if Re_s < 100 else 1.35
    Fb = _bypass_factor(alpha, Ab_As, ratio_strips)               # Eq. 23
    if Re_s > 2000:
        Fn = 1.0
    elif Re_s > 100:
        Fn = float(np.interp(Ncv, *FN_FIG9))                      # Fig 9
    else:
        Fn = (10.0 / Ncv) ** 0.18                                 # Eq. 21
    if cfg.Fw_method == "taborek":
        Fw = 0.55 + 0.72 * Fc                                     # Eq. 22
    else:
        Fw = float(np.interp(Rw, *FW_FIG10))                      # Fig 10
    if cfg.FL_method == "taborek":
        rs = Asb / (Atb + Asb)
        rlm = (Atb + Asb) / As
        FL = 0.44 * (1 - rs) + (1 - 0.44 * (1 - rs)) * math.exp(-2.2 * rlm)   # Eq. 25
    else:
        betaL = float(np.interp(AL_As, *BETA_L_FIG13))
        FL = 1 - betaL * (Atb + 2 * Asb) / AL                     # Eq. 24
    Fprod = Fn * Fw * Fb * FL

    # ---------------- Steps 3,5,6,7 with wall-viscosity iteration ----------------
    mu_w_t = tube.mu_w or tube.mu
    mu_w_s = shell.mu_w or shell.mu
    kw = ctx["kw"]
    Rfo, Rfi = shell.Rf, tube.Rf
    n_wall = cfg.wall_iterations if (tube.mu_of_T or shell.mu_of_T) else 1
    for _ in range(n_wall):
        # tube side, Eq. 7
        if Re_t < 2100:
            Nu = 1.86 * (Re_t * tube.Pr * di / Leff) ** (1 / 3) * (tube.mu / mu_w_t) ** 0.14
            lam = True
            if Nu < 3.66:
                W.append("Laminar tube-side Nu < 3.66 (document's Sieder-Tate form used as given)")
        else:
            Nu = 0.023 * Re_t ** 0.8 * tube.Pr ** 0.33 * (tube.mu / mu_w_t) ** 0.14
            lam = False
        hi = Nu * tube.k / di
        # shell side, Eq. 19 and 20
        hoc = jh * shell.k / do * Re_s * shell.Pr ** (1 / 3) * (shell.mu / mu_w_s) ** 0.14
        hs = hoc * Fprod
        # overall, Eq. 26
        R_wall = do * math.log(do / di) / (2 * kw)
        invU = 1 / hs + Rfo + R_wall + do / di * (Rfi + 1 / hi)
        Uo = 1 / invU
        # New wall viscosities (Step 9: recompute mu_w from wall temperatures).
        # The wall temperatures must be split out of the SAME temperature difference that the
        # mean bulk temperatures imply, not out of Ft*dTlm - otherwise Tw_s - Tw_t does not
        # equal q*R_wall and the two walls are inconsistent with each other.
        dT_bulk = shell.T_mean - tube.T_mean          # signed: positive when the shell is hot
        q = Uo * dT_bulk                              # mean flux on the outside area, signed
        Tw_s = shell.T_mean - q * (1 / hs + Rfo)
        Tw_t = tube.T_mean + q * (do / di) * (1 / hi + Rfi)
        if shell.mu_of_T:
            mu_w_s = shell.mu_of_T(Tw_s)
        if tube.mu_of_T:
            mu_w_t = tube.mu_of_T(Tw_t)

    # ---------------- Step 3: tube-side pressure drop, Eq. 8 ----------------
    if lam:
        jf_t, m_exp = 8.0 / Re_t, 0.25
    else:
        jf_t, m_exp = 0.0395 * Re_t ** -0.25, 0.14
    dPt_pass = Np * (8 * jf_t * L / di * (tube.mu / mu_w_t) ** (-m_exp) + 2.5) * tube.rho * ut ** 2 / 2
    dPt_noz, d_noz_t, u_noz_t = _nozzle_dp(tube.m, tube.rho, cfg.nozzle_d_tube, cfg.nozzle_rho_v2)
    dPt = n * (dPt_pass + dPt_noz)
    if 2100 <= Re_t < 4000:
        W.append("Tube-side Re in transition (2100-4000): turbulent line used as conservative estimate")

    # ---------------- Step 8: shell-side pressure drop ----------------
    jf = jf_shell(Re_s, layout, Pt / do)                                         # Eq. 28
    dPi = 8 * jf * Ncv * shell.rho * us ** 2 / 2 * (shell.mu / mu_w_s) ** -0.14   # Eq. 27
    alpha_p = 5.0 if Re_s < 100 else 4.0
    Fbp = _bypass_factor(alpha_p, Ab_As, ratio_strips)                           # F'b
    betaLp = float(np.interp(AL_As, *BETA_LP_FIG13))
    FLp = 1 - betaLp * (Atb + 2 * Asb) / AL                                      # Eq. 30
    dPc = dPi * Fbp * FLp                                                        # Eq. 29
    Nwv = 0.8 * cfg.Bc * Ds / Ppl
    uw = Ws / (shell.rho * Aw_net)
    uz = math.sqrt(uw * us)
    dPw = FLp * (2 + 0.6 * Nwv) * shell.rho * uz ** 2 / 2                        # Eq. 31
    dPe = dPi * (1 + Nwv / Ncv) * Fbp                                            # Eq. 32
    dPs_body = 2 * dPe + dPc * (Nb - 1) + Nb * dPw                               # Eq. 33
    dPs_noz, d_noz_s, u_noz_s = _nozzle_dp(Ws, shell.rho, cfg.nozzle_d_shell, cfg.nozzle_rho_v2)
    dPs = n * (dPs_body + dPs_noz)
    if Re_s < 100:
        W.append("Shell Re < 100: window dP (Eq. 31) not valid; use laminar window term (Bell / Coulson & Richardson)")

    # ---------------- Step 7 / 9 checks ----------------
    # Both readings of Step 7 (they differ because Nt is whole and/or Ds may be rounded to a standard size):
    margin_area = Uo / Uo_req - 1.0          # A_provided / A_required(Uo_calc) - 1
    margin_Uo = Uo / Uo_assumed - 1.0        # "Uo calculated exceeds Uo assumed by 10-25%"
    margin = margin_Uo if cfg.accept_criterion == "Uo" else margin_area
    if Fprod < 0.5:
        W.append(f"Fn*Fw*Fb*FL = {Fprod:.2f} < 0.5: too much bypass/leakage")
    if cfg.liquid_velocity_check and not (1.0 <= ut <= 2.5):
        W.append(f"Tube velocity {ut:.2f} m/s outside 1-2.5 m/s (change passes / tubes)")
    if cfg.liquid_velocity_check and not (0.3 <= us <= 1.0):
        W.append(f"Shell cross-flow velocity {us:.2f} m/s outside 0.3-1 m/s")
    if Ft < 0.8:
        W.append(f"Ft = {Ft:.3f} < 0.8 even with {n} shells")
    if not (cfg.margin_lo <= margin <= cfg.margin_hi):
        W.append(f"{'Uo' if cfg.accept_criterion == 'Uo' else 'Area'} margin {margin * 100:.1f}% outside "
                 f"{cfg.margin_lo * 100:.0f}-{cfg.margin_hi * 100:.0f}% window "
                 f"(area {margin_area * 100:.1f}%, Uo {margin_Uo * 100:.1f}%)")
    if dPs > cfg.dP_shell_max:
        W.append(f"Shell dP {dPs / 1e3:.1f} kPa > allowable {cfg.dP_shell_max / 1e3:.0f} kPa")
    if dPt > cfg.dP_tube_max:
        W.append(f"Tube dP {dPt / 1e3:.1f} kPa > allowable {cfg.dP_tube_max / 1e3:.0f} kPa")

    feasible = (margin >= 0 and dPs <= cfg.dP_shell_max and dPt <= cfg.dP_tube_max and Ft >= 0.8
                and (not cfg.liquid_velocity_check or 1.0 <= ut <= 2.5))
    return dict(
        ok=True, feasible=feasible, warnings=W, Uo_assumed=Uo_assumed,
        # geometry
        n_shells=n, do=do, di=di, t=t, Pt=Pt, layout=layout, passes=Np, L=L, L_eff=Leff,
        tubesheet_mm=ctx['ts_mm'], tubesheet_min_mm=ctx['ts_min_mm'], n_tubesheets=ctx['n_ts'], Ds=Ds, Db=Db, Nt=Nt,
        clearance=cl, lB=lB, Bc=cfg.Bc, Nb=Nb, A_prov_total=A_prov_total, A_req_total=A_req_total,
        L_over_Ds=L / Ds,
        # shell geometry
        As=As, Ncv=Ncv, Aw=Aw, Aw_net=Aw_net, Fc=Fc, Rw=Rw, Nw=Nw, Ab=Ab, Ab_lanes=Ab_lanes,
        n_pass_lanes=n_lanes, Atb=Atb, Asb=Asb, AL=AL,
        Ab_As=Ab_As, AL_As=AL_As, Nwv=Nwv, delta_sb=delta_sb,
        # tube side
        u_t=ut, Re_t=Re_t, h_i=hi, dP_t=dPt, dP_t_tubes=n * dPt_pass, dP_t_nozzles=n * dPt_noz,
        d_noz_t=d_noz_t, u_noz_t=u_noz_t, d_noz_s=d_noz_s, u_noz_s=u_noz_s,
        # shell side
        Gs=Gs, u_s=us, Re_s=Re_s, jh=jh, jf=jf, h_oc=hoc, Fn=Fn, Fw=Fw, Fb=Fb, FL=FL, Fprod=Fprod, h_s=hs,
        dPi=dPi, Fbp=Fbp, FLp=FLp, dPc=dPc, dPw=dPw, dPe=dPe, dP_s=dPs, dP_s_nozzles=n * dPs_noz,
        # overall
        Uo=Uo, Uo_req=Uo_req, margin=margin, margin_area=margin_area, margin_Uo=margin_Uo,
        mu_w_t=mu_w_t, mu_w_s=mu_w_s, Tw_s=Tw_s, Tw_t=Tw_t, R_wall=R_wall,
    )


# =============================================================================
# DESIGN LOOP (Figure 1) AND SEARCH (Step 9)
# =============================================================================
def build_context(hot: Fluid, cold: Fluid, cfg: Config) -> dict:
    Q, notes = resolve_duty(hot, cold)
    dTlm = lmtd(hot, cold)
    R, P = R_P(hot, cold)
    n, Ft = choose_shells(R, P, cfg)
    if cfg.do_mm not in TUBES_MM:
        raise ValueError(f"OD must be one of {sorted(TUBES_MM)} (Table 2)")
    kw = cfg.kw if cfg.kw is not None else WALL_K_STAINLESS
    t = TUBES_MM[cfg.do_mm] * 1e-3
    shell, tube = (cold, hot) if cfg.hot_in_tubes else (hot, cold)
    fouls = cfg.shell_fouls if cfg.shell_fouls is not None else shell.Rf > 0
    Pt_min = MIN_PITCH_MM[cfg.do_mm][1 if fouls else 0]
    Pt_mm = cfg.Pt_mm if cfg.Pt_mm else Pt_min
    Pt = Pt_mm * 1e-3
    notes_w = list(notes)
    # TEMA RCB-7.11 tubesheet thickness
    ts_min, ts_net = min_tubesheet_mm(cfg.do_mm, cfg.tema_class, cfg.corrosion_allowance_mm,
                                      cfg.tubesheet_round_mm)
    if cfg.tubesheet_mm is not None and cfg.tubesheet_mm < ts_min - 1e-9:
        raise ValueError(f"Tubesheet {cfg.tubesheet_mm:.1f} mm is below the TEMA {cfg.tema_class.upper()}-7.11 "
                         f"minimum of {ts_min:.1f} mm (net {ts_net:.1f} mm + CA {cfg.corrosion_allowance_mm:.1f} mm)")
    ts_mm = cfg.tubesheet_mm if cfg.tubesheet_mm is not None else ts_min
    n_ts = TUBESHEETS_PER_HEAD[cfg.head]
    ts_loss = n_ts * ts_mm * 1e-3 if cfg.tubesheet_deducts_length else 0.0
    if cfg.L - ts_loss <= 0:
        raise ValueError("Tube length is not longer than the tubesheet thickness")
    notes_w.append(f"Tubesheet TEMA {cfg.tema_class.upper()}-7.11: min {ts_min:.1f} mm (net {ts_net:.1f} + CA "
                   f"{cfg.corrosion_allowance_mm:.1f}), used {ts_mm:.1f} mm x {n_ts}; "
                   f"effective tube length {cfg.L - ts_loss:.3f} m of {cfg.L:.3f} m")
    # TEMA Pt/do >= 1.25.  Table 3 itself is rounded to 2 dp and two of its clean-service
    # entries sit a whisker below 1.25 (19.84/15.88 = 1.2494, 23.81/19.05 = 1.2499), so the
    # test needs a tolerance or the document's own default pitch is rejected.
    if Pt_mm < 1.25 * cfg.do_mm * (1 - 2e-3):
        raise ValueError(f"TEMA requires Pt/do >= 1.25 (got Pt = {Pt_mm:.2f} mm, "
                         f"Pt/do = {Pt_mm / cfg.do_mm:.4f})")
    if Pt_mm < 1.25 * cfg.do_mm:
        notes_w.append(f"Pt/do = {Pt_mm / cfg.do_mm:.4f}: Table 3 rounding, treated as the 1.25 minimum")
    if fouls and Pt_mm < cfg.do_mm + 6.35 - 1e-9:
        notes_w.append(f"Fouling shell side but Pt = {Pt_mm:.2f} mm < do + 6.35 = {cfg.do_mm + 6.35:.2f} mm: "
                       "bundle cannot be mechanically cleaned (Step 2b)")
    if fouls and cfg.layout == 30:
        notes_w.append("Shell side fouls: 90 or 45 deg layout is preferred for mechanical cleaning (Step 2b)")
    if cfg.do_mm == 25.40:
        notes_w.append("25.40 mm OD used (t = 2.77 -> di = 19.86 mm); the lecture tables list this tube "
                       "as 24.40 / 25.04 mm - confirm against your course table")
    if cfg.Fw_method == "chart" or cfg.FL_method == "chart":
        notes_w.append("Fw / betaL read from charts digitized by eye (+/-0.02); for a graded assignment "
                       "read the original lecture figures")
    return dict(ts_mm=ts_mm, ts_min_mm=ts_min, ts_net_mm=ts_net, n_ts=n_ts, L_eff=cfg.L - ts_loss,
                Q=Q, dTlm=dTlm, R=R, P=P, Ft=Ft, n_shells=n, t=t, kw=kw, Pt=Pt, fouls=fouls,
                tube=tube, shell=shell, shell_is_hot=not cfg.hot_in_tubes, notes=notes_w,
                hot=hot, cold=cold)


def design(hot: Fluid, cold: Fluid, cfg: Config, verbose: bool = False) -> dict:
    """Run the Figure-1 loop: assume Uo -> geometry -> Uo_calc -> revise Uo until the area margin
    is inside the window (or the standard-size choice starts cycling)."""
    hot = replace(hot)
    cold = replace(cold)          # copies, because resolve_duty fills in missing values
    ctx = build_context(hot, cold, cfg)
    history, seen = [], set()
    Uo = cfg.Uo_assumed
    target = 0.5 * (cfg.margin_lo + cfg.margin_hi)
    for it in range(cfg.max_iter):
        res = evaluate(cfg, ctx, Uo)
        if not res["ok"]:
            res.update(ctx=ctx, cfg=cfg, history=history, feasible=False)
            return res
        history.append(res)
        if verbose:
            print(f"  iter {it + 1}: Uo_assumed={Uo:7.1f}  Ds={res['Ds'] * 1e3:7.1f} mm  Nt={res['Nt']:5d}  "
                  f"Uo_calc={res['Uo']:7.1f}  margin={res['margin'] * 100:6.1f}%")
        if cfg.margin_lo <= res["margin"] <= cfg.margin_hi:
            break
        key = (res["Ds"], res["Nt"])
        if key in seen:
            # Revising Uo alone cannot move the design any further: the tube count (and, with
            # round_shell_to_standard, the shell diameter) is quantized, so the margin jumps in steps
            # and the loop would sit on the same design for ever.  With standard shells, walk the
            # neighbouring sizes (_shell_walk returns [] when Ds is not a standard size).
            history += _shell_walk(cfg, ctx, Uo, res)
            break
        seen.add(key)
        Uo = res["Uo"] / (1 + target)
    # best = inside the margin window if any such design exists, else smallest area with
    # margin >= 0, else the largest margin available
    inside = [r for r in history if cfg.margin_lo <= r["margin"] <= cfg.margin_hi]
    pos = [r for r in history if r["margin"] >= 0]
    if inside:
        best = min(inside, key=lambda r: r["A_prov_total"])
    elif pos:
        best = min(pos, key=lambda r: r["A_prov_total"])
    else:
        best = max(history, key=lambda r: r["margin"])
    best = dict(best)
    best.update(ctx=ctx, cfg=cfg, history=history, iterations=len(history),
                converged=cfg.margin_lo <= best["margin"] <= cfg.margin_hi)
    return best


def _shell_walk(cfg: Config, ctx: dict, Uo: float, res: dict, max_steps: int = 6) -> list:
    """Step up or down the standard shell list until the margin enters the window (or crosses it).
    Returns the extra trials, in order."""
    try:
        idx = STD_SHELLS_M.index(res["Ds"])
    except ValueError:
        return []
    out = []
    step = -1 if res["margin"] > cfg.margin_hi else +1     # too much area -> smaller shell
    for k in range(1, max_steps + 1):
        j = idx + step * k
        if not (0 <= j < len(STD_SHELLS_M)):
            break
        r = evaluate(cfg, ctx, Uo, force_Ds=STD_SHELLS_M[j])
        if not r.get("ok"):
            break
        out.append(r)
        if cfg.margin_lo <= r["margin"] <= cfg.margin_hi:
            break
        if step < 0 and r["margin"] < cfg.margin_lo:       # overshot: no smaller shell will do
            break
        if step > 0 and r["margin"] > cfg.margin_hi:
            break
    return out


def sweep(hot: Fluid, cold: Fluid, base: Config, *, do_mm=(19.05,), passes=(2, 4), layouts=(30, 90),
          L=(3.66, 4.88, 6.1), Bc=(0.25, 0.30), lB_frac=(0.2, 0.3, 0.4), top=10):
    """Step 9 iteration rules automated: try combinations, keep feasible ones, rank by area.
    Returns (best designs, number feasible, failures) - failures are NOT swallowed silently,
    each one is reported as (config summary, reason)."""
    out, failures = [], []
    for d, p, lay, l, bc, lf in itertools.product(do_mm, passes, layouts, L, Bc, lB_frac):
        cfg = replace(base, do_mm=d, passes=p, layout=lay, L=l, Bc=bc, lB_frac=lf)
        tag = f"do={d} passes={p} layout={lay} L={l} Bc={bc} lB_frac={lf}"
        try:
            r = design(hot, cold, cfg)
        except Exception as e:
            failures.append((tag, f"{type(e).__name__}: {e}"))
            continue
        if not r.get("ok"):
            failures.append((tag, r.get("reason", "not ok")))
        elif r.get("feasible"):
            out.append(r)
    out.sort(key=lambda r: (r["A_prov_total"], r["dP_s"] + r["dP_t"]))
    return out[:top], len(out), failures


# =============================================================================
# REPORT
# =============================================================================
def report(r: dict) -> str:
    ctx, cfg = r["ctx"], r["cfg"]
    if not r.get("ok", True):
        return f"DESIGN FAILED: {r['reason']}"
    tube, shell = ctx["tube"], ctx["shell"]
    L = []
    a = L.append
    a("=" * 74)
    a("BELL-DELAWARE SHELL & TUBE DESIGN - RESULTS")
    a("=" * 74)
    a(f"Duty Q = {ctx['Q'] / 1e3:,.1f} kW   LMTD = {ctx['dTlm']:.2f} K   R = {ctx['R']:.3f}   P = {ctx['P']:.3f}")
    a(f"Ft = {ctx['Ft']:.3f}   shells in series = {ctx['n_shells']}   "
      f"tube side: {tube.name}   shell side: {shell.name}")
    for n_ in ctx["notes"]:
        a(f"  note: {n_}")
    a("-" * 74)
    a("FINAL DESIGN TABLE (per shell unless stated)")
    a(f"  Shell ID Ds             {r['Ds'] * 1e3:8.1f} mm      Bundle Db      {r['Db'] * 1e3:8.1f} mm")
    a(f"  Tubes Nt                {r['Nt']:8d}         Tube OD x t    {r['do'] * 1e3:.2f} x {r['t'] * 1e3:.2f} mm")
    a(f"  Tube length L           {r['L']:8.2f} m       Layout / pitch {r['layout']} deg / {r['Pt'] * 1e3:.2f} mm")
    a(f"  Effective length Leff   {r['L_eff']:8.3f} m       Tubesheet      {r['tubesheet_mm']:8.1f} mm x {r['n_tubesheets']}"
      f"  (TEMA {cfg.tema_class.upper()}-7.11 min {r['tubesheet_min_mm']:.1f}, CA {cfg.corrosion_allowance_mm:.1f})")
    a(f"  Tube passes             {r['passes']:8d}         L/Ds           {r['L_over_Ds']:8.1f}")
    a(f"  Baffle spacing lB       {r['lB'] * 1e3:8.0f} mm      Baffle cut Bc  {r['Bc'] * 100:8.0f} %")
    a(f"  Baffles Nb              {r['Nb']:8d}         Sealing pairs  {cfg.n_seal_pairs:8d}")
    a(f"  Pass-partition lanes    {r['n_pass_lanes']:8d}         Lane width     {cfg.pass_lane_mm:8.2f} mm")
    a("-" * 74)
    a("TUBE SIDE")
    a(f"  velocity {r['u_t']:.2f} m/s   Re = {r['Re_t']:,.0f}   h_i = {r['h_i']:.0f} W/m2K   "
      f"dP_t = {r['dP_t'] / 1e3:.2f} kPa (nozzles {r['dP_t_nozzles'] / 1e3:.2f}, "
      f"d = {r['d_noz_t'] * 1e3:.0f} mm at {r['u_noz_t']:.1f} m/s)")
    a("SHELL SIDE")
    a(f"  As = {r['As']:.4f} m2  Ncv = {r['Ncv']:.1f}  Fc = {r['Fc']:.3f}  Rw = {r['Rw']:.3f}  "
      f"Ab/As = {r['Ab_As']:.3f} (lanes {r['Ab_lanes'] / r['Ab'] * 100:.0f}% of Ab)  AL/As = {r['AL_As']:.3f}")
    a(f"  cross-flow velocity {r['u_s']:.3f} m/s   Re = {r['Re_s']:,.0f}   jh = {r['jh']:.4g}   jf = {r['jf']:.4g}")
    a(f"  h_oc = {r['h_oc']:.0f}   Fn={r['Fn']:.3f} Fw={r['Fw']:.3f} Fb={r['Fb']:.3f} FL={r['FL']:.3f}  "
      f"(product {r['Fprod']:.3f})   h_s = {r['h_s']:.0f} W/m2K")
    a(f"  dPi={r['dPi']:.0f}  dPc={r['dPc']:.0f}  dPw={r['dPw']:.0f}  dPe={r['dPe']:.0f} Pa (per zone)  "
      f"-> dP_s = {r['dP_s'] / 1e3:.2f} kPa (nozzles {r['dP_s_nozzles'] / 1e3:.2f}, "
      f"d = {r['d_noz_s'] * 1e3:.0f} mm at {r['u_noz_s']:.1f} m/s)")
    a("-" * 74)
    a("OVERALL")
    a(f"  Uo assumed = {r['Uo_assumed']:.0f}   Uo calculated = {r['Uo']:.0f}   Uo required for provided area = {r['Uo_req']:.0f} W/m2K")
    a(f"  Area required (at assumed Uo) = {r['A_req_total']:.2f} m2   provided = {r['A_prov_total']:.2f} m2")
    a(f"  Step 7 margins: area {r['margin_area'] * 100:+.1f} %   Uo_calc vs Uo_assumed {r['margin_Uo'] * 100:+.1f} %"
      f"   (driving criterion: {cfg.accept_criterion})")
    a(f"  wall temperatures: shell side {r['Tw_s']:.1f} degC, tube side {r['Tw_t']:.1f} degC "
      f"(difference {r['Tw_s'] - r['Tw_t']:+.2f} K across the wall)")
    a(f"  wall viscosities used: tube mu_w = {r['mu_w_t']:.3e}, shell mu_w = {r['mu_w_s']:.3e} Pa s")
    a(f"  iterations: {r.get('iterations', 1)}   converged to margin window: {r.get('converged')}   "
      f"feasible (margin>=0, dP ok, velocity ok): {r['feasible']}")
    if r["warnings"]:
        a("-" * 74)
        a("WARNINGS / CHECKS")
        for w in r["warnings"]:
            a(f"  ! {w}")
    a("=" * 74)
    return "\n".join(L)

In [ ]:
# ==============================================================================
# CELL 3: STREAM PREPARATION  (user inputs + property engine -> Bell-Delaware Fluid objects)
# ==============================================================================
# Your Fluid class needs rho, mu, cp, k "at mean temperature" and exactly one missing
# value (flow or outlet temperature) to solve from the duty.  This bridge:
#   * looks the properties up from Cell 1 at the stream's mean bulk temperature,
#   * iterates because the mean temperature itself depends on the (possibly unknown) outlet,
#   * treats an input of 0 as "unknown" (same convention as the PHE dashboard),
#   * applies the PHE "assumed effectiveness" rule when BOTH outlets are unknown,
#   * hands each stream a mu_of_T() so your wall-temperature iteration (Step 9) works
#     with the same property model.
# ==============================================================================

def _none_if_unknown(v):
    return None if (v is None or float(v) <= 0.0) else float(v)


def _make_mu_of_T(key: str, P_bar: float):
    """Viscosity-vs-temperature callable for Fluid.mu_of_T (no phase check at the wall)."""
    return lambda T, _k=key, _p=P_bar: get_fluid_properties(_k, T, _p, check_phase=False)[3]


def _build_fluid(spec: dict, T_mean: float) -> Fluid:
    key, P = spec["fluid"], spec.get("P_bar", 2.0)
    rho, cp, k, mu = get_fluid_properties(key, T_mean, P)
    return Fluid(name=spec.get("label", key), T_in=float(spec["T_in"]),
                 T_out=_none_if_unknown(spec.get("T_out")), m=_none_if_unknown(spec.get("m")),
                 rho=rho, mu=mu, cp=cp, k=k, Rf=float(spec.get("Rf", 0.0)),
                 mu_of_T=_make_mu_of_T(key, P))


def prepare_streams(hot_spec: dict, cold_spec: dict, eps_assumed: float = 0.85,
                    max_iter: int = 12, tol: float = 0.01):
    """
    hot_spec / cold_spec keys: label, fluid, T_in, T_out (<=0 -> unknown), m (<=0 -> unknown),
                               P_bar, Rf.
    Supported specifications (same logic as resolve_duty, plus the epsilon rule):
      * one stream fully known (flow + both temperatures), other stream missing flow OR outlet
      * both flows known and both outlets unknown -> duty from assumed effectiveness eps
    Returns (hot, cold, notes, eps_used_or_None) with every field of both Fluids filled in and the
    properties evaluated at the converged mean bulk temperatures.
    """
    if float(hot_spec["T_in"]) <= float(cold_spec["T_in"]):
        raise ValueError("Hot inlet temperature must exceed cold inlet temperature.")
    Thm = float(hot_spec["T_in"]) - 10.0 if _none_if_unknown(hot_spec.get("T_out")) is None else \
        0.5 * (float(hot_spec["T_in"]) + float(hot_spec["T_out"]))
    Tcm = float(cold_spec["T_in"]) + 10.0 if _none_if_unknown(cold_spec.get("T_out")) is None else \
        0.5 * (float(cold_spec["T_in"]) + float(cold_spec["T_out"]))
    notes, eps_used = [], None
    for _ in range(max_iter):
        h, c = _build_fluid(hot_spec, Thm), _build_fluid(cold_spec, Tcm)
        eps_used = None
        if h.T_out is None and c.T_out is None and h.m is not None and c.m is not None:
            Cmin = min(h.m * h.cp, c.m * c.cp)
            Q = eps_assumed * Cmin * (h.T_in - c.T_in)
            h.T_out = h.T_in - Q / (h.m * h.cp)
            c.T_out = c.T_in + Q / (c.m * c.cp)
            eps_used = eps_assumed
        try:
            Q, resolve_notes = resolve_duty(h, c)
        except ValueError as e:
            raise ValueError(
                f"{e}  Give flow + both temperatures for one stream and flow OR outlet for the other "
                "(or both flows with both outlets = 0 to use the assumed effectiveness).")
        if None in (h.m, h.T_out, c.m, c.T_out):
            raise ValueError("Under-specified duty: could not resolve all flows and temperatures.")
        if Q <= 0:
            raise ValueError(f"Non-positive duty ({Q / 1e3:.1f} kW): check the inlet/outlet temperatures.")
        Thm_new, Tcm_new = h.T_mean, c.T_mean
        done = abs(Thm_new - Thm) < tol and abs(Tcm_new - Tcm) < tol
        Thm, Tcm = Thm_new, Tcm_new
        if done:
            break
    if eps_used is not None:
        notes.append(f"Only inlet temperatures given: assumed effectiveness eps = {eps_used:.2f} "
                     f"-> Q = {Q / 1e3:,.1f} kW, outlets {h.T_out:.1f} / {c.T_out:.1f} degC")
    return h, c, notes, eps_used


def stream_property_table(hot: Fluid, cold: Fluid) -> str:
    rows = [f"  {'stream':<28}{'Tmean':>7}{'rho':>9}{'cp':>8}{'k':>8}{'mu [mPa s]':>12}{'Pr':>7}{'Rf':>10}"]
    for f in (hot, cold):
        rows.append(f"  {f.name:<28}{f.T_mean:7.1f}{f.rho:9.1f}{f.cp:8.0f}{f.k:8.3f}{f.mu * 1e3:12.3f}{f.Pr:7.2f}{f.Rf:10.2e}")
    return "\n".join(rows)


print("Stream preparation bridge ready.")

Stream preparation bridge ready.


In [ ]:
# ==============================================================================
# CELL 4: BENCHMARK & VERIFICATION SUITE
# ==============================================================================
# A. Your original water-water benchmark, exactly as in bell_delaware_v3.py
#    (hard-coded properties, fixed temperatures) -> regression reference.
# B. The SAME duty and geometry, but with fluid properties from the Cell-1 engine
#    evaluated at the mean temperatures -> shows the property bridge reproduces A.
# C. Water property model vs CoolProp (IAPWS-95).
# D. Multi-fluid smoke test (ethanol-water, CoolProp ethanol, oil) against Table 1 bands.
#    This checks that the property bridge works for other fluids; it is a range check,
#    NOT a validation against a published design.
# ==============================================================================

# ---- A. original benchmark (verbatim from your script) ----------------------------------
def water_mu(T_c: float) -> float:
    """Simple water viscosity correlation (Pa s) for the demo; T in degC."""
    return 2.414e-5 * 10 ** (247.8 / (T_c + 273.15 - 140.0))


def benchmark_original():
    hot = Fluid("hot water", T_in=80, T_out=50, m=None, rho=980, mu=water_mu(65), cp=4185, k=0.66,
                Rf=1.76e-4, mu_of_T=water_mu)
    # cold water 40 kg/s, 25 -> 40 degC  (Q = 2.51 MW, solved from the hot side if m is given)
    hot.m = 20.0
    cold = Fluid("cold water", T_in=25, T_out=None, m=40.0, rho=995, mu=water_mu(32.5), cp=4178, k=0.62,
                 Rf=1.76e-4, mu_of_T=water_mu)
    return hot, cold


def print_sweep_table(best):
    print(f"{'Ds mm':>6} {'Nt':>5} {'pass':>4} {'lay':>3} {'L m':>5} {'Bc':>5} {'lB mm':>6} "
          f"{'A m2':>6} {'Uo':>5} {'mar%':>5} {'dPs kPa':>7} {'dPt kPa':>7}")
    for r in best:
        print(f"{r['Ds'] * 1e3:6.1f} {r['Nt']:5d} {r['passes']:4d} {r['layout']:3d} {r['L']:5.2f} {r['Bc']:5.2f} "
              f"{r['lB'] * 1e3:6.0f} {r['A_prov_total']:6.1f} {r['Uo']:5.0f} {r['margin'] * 100:5.1f} "
              f"{r['dP_s'] / 1e3:7.1f} {r['dP_t'] / 1e3:7.1f}")


BENCH_CFG = Config(hot_in_tubes=True, Uo_assumed=1000, do_mm=19.05, layout=90,
                   passes=2, L=4.88, head="fixed", Bc=0.25, lB_frac=0.3)


def run_benchmark_suite(verbose_original: bool = True):
    print("=" * 100)
    print("SHELL & TUBE (BELL-DELAWARE) VERIFICATION SUITE")
    print("=" * 100)

    # ---------------- A ----------------
    print("\n[A] Original water-water benchmark (your script, fixed properties)")
    print("    Table 1 expects Uo = 800-1500 W/m2K for water-water.")
    hotA, coldA = benchmark_original()
    resA = design(hotA, coldA, BENCH_CFG, verbose=verbose_original)
    print(report(resA))
    print("\n    Step 9 automated search over passes / layout / L / Bc / baffle spacing ...")
    best, n_ok, fails = sweep(hotA, coldA, BENCH_CFG)
    if fails:
        print(f"    {len(fails)} combinations could not be designed, e.g. {fails[0][0]} -> {fails[0][1]}")
    print(f"    {n_ok} feasible designs; best by area:")
    print_sweep_table(best)

    # ---------------- B ----------------
    print("\n" + "-" * 100)
    print("[B] Same duty & geometry, properties from the Cell-1 engine at mean temperatures")
    hs = dict(label="hot water", fluid="water", T_in=80, T_out=50, m=20, P_bar=2.0, Rf=1.76e-4)
    cs = dict(label="cold water", fluid="water", T_in=25, T_out=0, m=40, P_bar=2.0, Rf=1.76e-4)   # 0 = unknown
    hotB, coldB, notes, _ = prepare_streams(hs, cs)
    print(stream_property_table(hotB, coldB))
    resB = design(hotB, coldB, BENCH_CFG)
    keys = [("Ds [mm]", "Ds", 1e3), ("Nt", "Nt", 1), ("h_i [W/m2K]", "h_i", 1), ("h_s [W/m2K]", "h_s", 1),
            ("Uo [W/m2K]", "Uo", 1), ("dP_s [kPa]", "dP_s", 1e-3), ("dP_t [kPa]", "dP_t", 1e-3),
            ("area margin [%]", "margin", 100)]
    print(f"\n    {'quantity':<18}{'[A] original':>14}{'[B] engine props':>18}{'diff %':>10}")
    for lab, k, s in keys:
        a, b = resA[k] * s, resB[k] * s
        print(f"    {lab:<18}{a:14.2f}{b:18.2f}{(b / a - 1) * 100 if a else float('nan'):10.2f}")
    drift = abs(resB["Uo"] / resA["Uo"] - 1) * 100
    print(f"    -> Uo differs by {drift:.2f} % : property-model differences only "
          f"({'PASS' if drift < 3.0 else 'CHECK'}, tolerance 3 %)")

    # ---------------- C ----------------
    print("\n" + "-" * 100)
    if COOLPROP_AVAILABLE:
        print("[C] Water property model vs CoolProp (IAPWS-95, 10 bar liquid)")
        print(f"    {'T [C]':>6} | {'rho err%':>9} {'cp err%':>9} {'k err%':>9} {'mu err%':>9}")
        worst = [0.0] * 4
        for T in (5, 20, 40, 60, 80, 100, 120, 140):
            mine = water_properties(T)
            ref = [CP.PropsSI(p, "T", T + 273.15, "P", 10e5, "Water") for p in ("D", "C", "L", "V")]
            errs = [(m / r_ - 1) * 100 for m, r_ in zip(mine, ref)]
            worst = [max(w, abs(e)) for w, e in zip(worst, errs)]
            print(f"    {T:6d} | {errs[0]:9.3f} {errs[1]:9.3f} {errs[2]:9.3f} {errs[3]:9.3f}")
        print(f"    worst |err| %: rho {worst[0]:.3f}  cp {worst[1]:.3f}  k {worst[2]:.3f}  mu {worst[3]:.3f}")
    else:
        print("[C] Skipped: CoolProp not installed (pip install CoolProp to enable this check).")

    # ---------------- D ----------------
    print("\n" + "-" * 100)
    print("[D] Multi-fluid smoke test (range check against Table 1, not a validation)")
    cases = [
        ("50% ethanol-water (shell) / water (tubes)", "ethanol50", dict(T_in=78, T_out=35, m=10),
         dict(T_in=20, T_out=0, m=15), False, 500, "water", "organic solvents"),
        ("Pure ethanol, CoolProp (shell) / water (tubes)", "CP::Ethanol", dict(T_in=70, T_out=35, m=8),
         dict(T_in=20, T_out=0, m=15), False, 500, "water", "organic solvents"),
        ("ISO VG 46 oil (shell) / water (tubes)", "oil", dict(T_in=70, T_out=45, m=5),
         dict(T_in=25, T_out=0, m=10), False, 300, "water", "light oils"),
    ]
    print(f"    {'case':<50}{'Q kW':>8}{'Ds mm':>8}{'Nt':>6}{'Uo':>7}   Table 1 band        status")
    for label, fkey, hd, cd, in_tubes, uo0, ckey, tkey in cases:
        if fkey.startswith("CP::") and not COOLPROP_AVAILABLE:
            print(f"    {label:<50}  skipped (CoolProp not installed)")
            continue
        h_, c_, _, _ = prepare_streams(
            dict(label=label.split(" (")[0], fluid=fkey, P_bar=2.0, Rf=1.76e-4, **hd),
            dict(label="cooling water", fluid="water" if not fkey.startswith("CP::") else "CP::Water",
                 P_bar=2.0, Rf=1.76e-4, **cd))
        cfg = replace(BENCH_CFG, hot_in_tubes=in_tubes, Uo_assumed=uo0)
        r = design(h_, c_, cfg)
        lo, hi = TABLE1_UO[(ckey, tkey)]
        if not r.get("ok"):
            print(f"    {label:<50}  design failed: {r['reason']}")
            continue
        inside = lo <= r["Uo"] <= hi
        print(f"    {label:<50}{r['ctx']['Q'] / 1e3:8.0f}{r['Ds'] * 1e3:8.0f}{r['Nt']:6d}{r['Uo']:7.0f}"
              f"   {lo:>4}-{hi:<4} W/m2K    {'inside' if inside else 'outside'}")
    print("    (Viscous-oil duties that are small relative to the shell size are flow-starved; a low Uo and "
          "velocity warnings there are expected - use Search mode to find a better layout.)")
    print("=" * 100)
    return resA, resB


benchmark_results = run_benchmark_suite()

SHELL & TUBE (BELL-DELAWARE) VERIFICATION SUITE

[A] Original water-water benchmark (your script, fixed properties)
    Table 1 expects Uo = 800-1500 W/m2K for water-water.
  iter 1: Uo_assumed= 1000.0  Ds=  557.0 mm  Nt=  296  Uo_calc= 1096.8  margin=  10.0%
  iter 2: Uo_assumed=  933.4  Ds=  573.7 mm  Nt=  317  Uo_calc= 1075.4  margin=  15.5%
BELL-DELAWARE SHELL & TUBE DESIGN - RESULTS
Duty Q = 2,511.0 kW   LMTD = 31.90 K   R = 1.997   P = 0.273
Ft = 0.920   shells in series = 1   tube side: hot water   shell side: cold water
  note: Tubesheet TEMA B-7.11: min 20.0 mm (net 14.3 + CA 0.0), used 20.0 mm x 2; effective tube length 4.840 m of 4.880 m
  note: Fw / betaL read from charts digitized by eye (+/-0.02); for a graded assignment read the original lecture figures
--------------------------------------------------------------------------
FINAL DESIGN TABLE (per shell unless stated)
  Shell ID Ds                573.7 mm      Bundle Db         560.1 mm
  Tubes Nt                     

In [ ]:
# ==============================================================================
# CELL 5: INTERACTIVE DASHBOARD & TELEMETRY PLOTS
# ==============================================================================
# Two operating modules (the S&T counterparts of the PHE Sizing / Rating tabs):
#   Module A - DESIGN : one geometry you specify, run through your Figure-1 loop (design()).
#   Module B - SEARCH : automated Step-9 sweep over passes / layout / L / Bc / baffle spacing (sweep()).
# Inputs mirror the PHE dashboard: fluid per stream (built-in / CoolProp / custom), temperatures,
# flows, pressure-drop limits, "0 = unknown" outlets with an assumed-effectiveness fallback,
# plus every Config option from your code (under "Advanced").
# ==============================================================================

import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 10
plt.rcParams['axes.linewidth'] = 1.2
plt.rcParams['grid.alpha'] = 0.5
plt.rcParams['grid.linestyle'] = '--'


# ------------------------------------------------------------------------------
# Plots
# ------------------------------------------------------------------------------
def plot_bell_results(r: dict, title: str):
    """2-panel report: pressure-drop audit, thermal-resistance split."""
    ctx, cfg = r["ctx"], r["cfg"]
    tube, shell = ctx["tube"], ctx["shell"]
    fig, axs = plt.subplots(1, 2, figsize=(12, 5), dpi=100)

    # Panel 1: pressure-drop audit ----------------------------------------------------
    ax2 = axs[0]
    labels = ["Tube \u0394P\n(actual)", "Tube limit\n(allowable)", "Shell \u0394P\n(actual)", "Shell limit\n(allowable)"]
    vals = [r["dP_t"] / 1e3, cfg.dP_tube_max / 1e3, r["dP_s"] / 1e3, cfg.dP_shell_max / 1e3]
    bars = ax2.bar(labels, vals, color=['#d62728', '#ff9896', '#1f77b4', '#aec7e8'],
                   edgecolor='black', lw=1.2, width=0.55)
    for b_ in bars:
        y = b_.get_height()
        ax2.text(b_.get_x() + b_.get_width() / 2.0, y + max(vals) * 0.01, f"{y:.1f} kPa",
                 ha='center', va='bottom', fontweight='bold', fontsize=9)
    ax2.set_title("Hydraulic Pressure-Drop Audit", fontsize=11, fontweight='bold')
    ax2.set_ylabel("Pressure drop [kPa]")
    ax2.set_ylim(0, max(vals) * 1.35)
    ax2.grid(True, axis='y')

    # Panel 2: thermal-resistance distribution (referred to the outside area) --------
    ax3 = axs[1]
    do, di = r["do"], r["di"]
    comps = [1.0 / r["h_s"], shell.Rf, r["R_wall"], do / di * tube.Rf, do / di / r["h_i"]]
    tot = sum(comps)
    fr = [c_ / tot * 100.0 for c_ in comps]
    names = ['Shell conv.\n(1/h_s)', 'Shell fouling\n(R_f,o)', 'Tube wall\n(ln term)',
             'Tube fouling\n(R_f,i\u00b7do/di)', 'Tube conv.\n(do/di/h_i)']
    bars3 = ax3.bar(names, fr, color=['#66b3ff', '#c2c2f0', '#99ff99', '#ffcc99', '#ff9999'],
                    edgecolor='black', lw=1.2, width=0.55)
    for b_ in bars3:
        y = b_.get_height()
        ax3.text(b_.get_x() + b_.get_width() / 2.0, y + 0.8, f"{y:.1f}%",
                 ha='center', va='bottom', fontweight='bold', fontsize=9)
    ax3.set_title(f"Thermal Resistance Distribution (Uo = {r['Uo']:.0f} W/m\u00b2K)", fontsize=11, fontweight='bold')
    ax3.set_ylabel("Share of total resistance (%)")
    ax3.set_ylim(0, max(fr) * 1.35)
    ax3.grid(True, axis='y')

    plt.tight_layout()
    plt.show()


# ------------------------------------------------------------------------------
# Default inputs (= your water-water benchmark) and the run function
# ------------------------------------------------------------------------------
DEFAULT_INPUTS = dict(
    mode="design",
    hot_fluid="water", hot_P=2.0, hot_Tin=80.0, hot_Tout=50.0, hot_m=20.0, hot_Rf=1.76e-4,
    cold_fluid="water", cold_P=2.0, cold_Tin=25.0, cold_Tout=0.0, cold_m=40.0, cold_Rf=1.76e-4,
    eps=0.85, hot_in_tubes=True, dP_shell_kPa=70.0, dP_tube_kPa=70.0,
    # geometry / Config
    Uo_assumed=1000.0, do_mm=19.05, layout=90, passes=2, L=4.88, head="fixed", Bc=0.25, lB_frac=0.3,
    Pt_mm=0.0, tema_class="B",
    # advanced
    corrosion_allowance_mm=0.0, tubesheet_mm=0.0, tubesheet_deducts_length=True, round_shell_to_standard=False,
    n_shells=0, n_seal_pairs=0, n_pass_lanes=-1, pass_lane_mm=19.05, Fw_method="chart", FL_method="chart",
    accept_criterion="area", margin_lo_pct=10.0, margin_hi_pct=25.0, LDs_lo=5.0, LDs_hi=10.0,
    liquid_velocity_check=True, delta_tb_mm=0.8, delta_sb_mm=0.0, nozzle_d_shell_mm=0.0, nozzle_d_tube_mm=0.0,
    kw=0.0,
    # search ranges
    s_do=(19.05,), s_passes=(2, 4), s_layouts=(30, 90), s_L=(3.66, 4.88, 6.1), s_Bc=(0.25, 0.30),
    s_lB=(0.2, 0.3, 0.4), s_top=10,
)


def config_from_inputs(p: dict) -> Config:
    return Config(
        hot_in_tubes=p["hot_in_tubes"], Uo_assumed=p["Uo_assumed"],
        kw=p["kw"] if p["kw"] > 0 else None, do_mm=p["do_mm"], layout=int(p["layout"]),
        Pt_mm=p["Pt_mm"] if p["Pt_mm"] > 0 else None, passes=int(p["passes"]), L=p["L"], head=p["head"],
        tema_class=p["tema_class"], corrosion_allowance_mm=p["corrosion_allowance_mm"],
        tubesheet_mm=p["tubesheet_mm"] if p["tubesheet_mm"] > 0 else None,
        round_shell_to_standard=p["round_shell_to_standard"],
        tubesheet_deducts_length=p["tubesheet_deducts_length"],
        n_shells=int(p["n_shells"]) if p["n_shells"] > 0 else None,
        Bc=p["Bc"], lB_frac=p["lB_frac"], delta_tb=p["delta_tb_mm"] * 1e-3,
        delta_sb=p["delta_sb_mm"] * 1e-3 if p["delta_sb_mm"] > 0 else None,
        n_seal_pairs=int(p["n_seal_pairs"]),
        n_pass_lanes=int(p["n_pass_lanes"]) if p["n_pass_lanes"] >= 0 else None,
        pass_lane_mm=p["pass_lane_mm"], Fw_method=p["Fw_method"], FL_method=p["FL_method"],
        nozzle_d_shell=p["nozzle_d_shell_mm"] * 1e-3 if p["nozzle_d_shell_mm"] > 0 else None,
        nozzle_d_tube=p["nozzle_d_tube_mm"] * 1e-3 if p["nozzle_d_tube_mm"] > 0 else None,
        accept_criterion=p["accept_criterion"], margin_lo=p["margin_lo_pct"] / 100.0,
        margin_hi=p["margin_hi_pct"] / 100.0, LDs_lo=p["LDs_lo"], LDs_hi=p["LDs_hi"],
        dP_shell_max=p["dP_shell_kPa"] * 1e3, dP_tube_max=p["dP_tube_kPa"] * 1e3,
        liquid_velocity_check=p["liquid_velocity_check"])


def _fluid_label(key: str) -> str:
    """Short display name for reports/legends, e.g. 'Water (IAPWS fits)' -> 'Water'."""
    import re
    for lab, k in fluid_menu() + [(n, n) for n in CUSTOM_FLUIDS]:
        if k == key:
            return re.sub(r"\s*\(.*\)\s*$", "", lab)
    return key


def run_bell_delaware(p: dict):
    """Runs one simulation from a plain dict of inputs (same keys as DEFAULT_INPUTS)."""
    p = {**DEFAULT_INPUTS, **p}
    hs = dict(label=f"hot {_fluid_label(p['hot_fluid'])}", fluid=p["hot_fluid"], T_in=p["hot_Tin"],
              T_out=p["hot_Tout"], m=p["hot_m"], P_bar=p["hot_P"], Rf=p["hot_Rf"])
    cs = dict(label=f"cold {_fluid_label(p['cold_fluid'])}", fluid=p["cold_fluid"], T_in=p["cold_Tin"],
              T_out=p["cold_Tout"], m=p["cold_m"], P_bar=p["cold_P"], Rf=p["cold_Rf"])
    print("=" * 85)
    print("            SHELL & TUBE (BELL-DELAWARE) THERMAL-HYDRAULIC SIMULATION")
    print("=" * 85)
    try:
        hot, cold, notes, eps_used = prepare_streams(hs, cs, p["eps"])
        cfg = config_from_inputs(p)
    except Exception as e:
        print(f"[INPUT ERROR] {e}")
        return None
    for n_ in notes:
        print(f"[INFO] {n_}")
    Q = hot.m * hot.cp * (hot.T_in - hot.T_out)
    print(f"Duty Q = {Q / 1e3:,.2f} kW   |   Hot: {hot.T_in:.1f} -> {hot.T_out:.2f} \u00b0C ({hot.m:.2f} kg/s)"
          f"   |   Cold: {cold.T_in:.1f} -> {cold.T_out:.2f} \u00b0C ({cold.m:.2f} kg/s)")
    print("Properties at mean bulk temperature:")
    print(stream_property_table(hot, cold))
    place = "hot in TUBES / cold in SHELL" if cfg.hot_in_tubes else "hot in SHELL / cold in TUBES"
    print(f"Allocation: {place}")
    print("-" * 85)

    try:
        if p["mode"] == "design":
            res = design(hot, cold, cfg, verbose=True)
            print(report(res))
            if res.get("ok", True) and "Ds" in res:
                plot_bell_results(res, f"Design: {res['passes']}-pass, Ds = {res['Ds'] * 1e3:.0f} mm, Nt = {res['Nt']}")
            return res
        # ---- search mode ----
        n_combo = (len(p["s_do"]) * len(p["s_passes"]) * len(p["s_layouts"]) * len(p["s_L"])
                   * len(p["s_Bc"]) * len(p["s_lB"]))
        print(f"Step 9 automated search: {n_combo} combinations "
              f"(OD {list(p['s_do'])}, passes {list(p['s_passes'])}, layouts {list(p['s_layouts'])}, "
              f"L {list(p['s_L'])}, Bc {list(p['s_Bc'])}, lB/Ds {list(p['s_lB'])}) ...")
        best, n_ok, fails = sweep(hot, cold, cfg, do_mm=tuple(p["s_do"]), passes=tuple(p["s_passes"]),
                                  layouts=tuple(p["s_layouts"]), L=tuple(p["s_L"]), Bc=tuple(p["s_Bc"]),
                                  lB_frac=tuple(p["s_lB"]), top=int(p["s_top"]))
        if fails:
            print(f"{len(fails)} combinations could not be designed, e.g. {fails[0][0]} -> {fails[0][1]}")
        print(f"{n_ok} feasible designs; best by area:")
        if not best:
            print("\u274c [SEARCH INFEASIBLE] No combination met margin >= 0, both pressure-drop limits, Ft >= 0.8 "
                  "and the tube-velocity window.")
            print("   Recommendation: relax the pressure-drop limits, widen the search ranges (more passes / "
                  "lengths), swap the stream allocation, or untick the liquid-velocity check.")
            return None
        print_sweep_table(best)
        print("\nFull specification of the top-ranked design (row 1 of the search):")
        print(report(best[0]))
        top = best[0]
        plot_bell_results(top, f"Search winner: {top['passes']}-pass, Ds = {top['Ds'] * 1e3:.0f} mm, Nt = {top['Nt']}")
        return best
    except Exception as e:
        print(f"[CALCULATION ERROR] {type(e).__name__}: {e}")
        return None


# ------------------------------------------------------------------------------
# Dashboard builder
# ------------------------------------------------------------------------------
def build_interactive_dashboard():
    """Constructs the ipywidgets GUI (Google Colab / Jupyter). Returns a dict of the widgets."""
    try:
        import ipywidgets as widgets
        from IPython.display import display, clear_output
    except ImportError:
        print("Note: ipywidgets not available - running the default benchmark case headless.")
        run_bell_delaware(dict(DEFAULT_INPUTS))
        return {}

    L_ = widgets.Layout
    desc_w = {'description_width': 'initial'}
    fluids = fluid_menu()

    w_mode = widgets.RadioButtons(
        options=[('Module A: Design Mode (one geometry, Figure-1 iteration)', 'design'),
                 ('Module B: Search Mode (automated Step-9 sweep)', 'search')],
        value='design', description='Module:', style=desc_w, layout=L_(width='650px'))

    # ---- hot stream -------------------------------------------------------------------
    w_hf = widgets.Dropdown(options=fluids, value="water", description='Hot Fluid:', layout=L_(width='300px'))
    w_hP = widgets.BoundedFloatText(value=2.0, min=0.1, max=300.0, step=0.5, description='P_hot [bar]:', layout=L_(width='260px'))
    w_thi = widgets.BoundedFloatText(value=80.0, min=-50.0, max=400.0, step=0.5, description='Hot T_in [\u00b0C]:', layout=L_(width='260px'))
    w_tho = widgets.FloatText(value=50.0, description='Hot T_out [\u00b0C] (0=?):', style=desc_w, layout=L_(width='260px'))
    w_mh = widgets.BoundedFloatText(value=20.0, min=0.0, max=5000.0, step=0.5, description='M_hot [kg/s] (0=?):', style=desc_w, layout=L_(width='260px'))
    w_rfh = widgets.BoundedFloatText(value=1.76e-4, min=0.0, max=1e-2, step=1e-5, description='Rf_hot [m\u00b2K/W]:', style=desc_w, layout=L_(width='260px'))

    # ---- cold stream ------------------------------------------------------------------
    w_cf = widgets.Dropdown(options=fluids, value="water", description='Cold Fluid:', layout=L_(width='300px'))
    w_cP = widgets.BoundedFloatText(value=2.0, min=0.1, max=300.0, step=0.5, description='P_cold [bar]:', layout=L_(width='260px'))
    w_tci = widgets.BoundedFloatText(value=25.0, min=-50.0, max=400.0, step=0.5, description='Cold T_in [\u00b0C]:', layout=L_(width='260px'))
    w_tco = widgets.FloatText(value=0.0, description='Cold T_out [\u00b0C] (0=?):', style=desc_w, layout=L_(width='260px'))
    w_mc = widgets.BoundedFloatText(value=40.0, min=0.0, max=5000.0, step=0.5, description='M_cold [kg/s] (0=?):', style=desc_w, layout=L_(width='260px'))
    w_rfc = widgets.BoundedFloatText(value=1.76e-4, min=0.0, max=1e-2, step=1e-5, description='Rf_cold [m\u00b2K/W]:', style=desc_w, layout=L_(width='260px'))

    # ---- allocation / limits ----------------------------------------------------------
    w_alloc = widgets.Dropdown(options=[('Hot fluid in TUBES, cold in SHELL', True), ('Hot fluid in SHELL, cold in TUBES', False)],
                               value=True, description='Allocation:', layout=L_(width='380px'))
    w_dps = widgets.BoundedFloatText(value=70.0, min=1.0, max=500.0, step=5.0, description='\u0394P_shell Max [kPa]:', style=desc_w, layout=L_(width='260px'))
    w_dpt = widgets.BoundedFloatText(value=70.0, min=1.0, max=500.0, step=5.0, description='\u0394P_tube Max [kPa]:', style=desc_w, layout=L_(width='260px'))

    # ---- custom fluid creator (same as the PHE dashboard) -----------------------------
    w_ca_name = widgets.Text(value="Dowtherm A", description="Fluid A:", layout=L_(width='220px'))
    w_ca_rho = widgets.FloatText(value=1056.0, description="\u03c1 [kg/m\u00b3]:", layout=L_(width='180px'))
    w_ca_cp = widgets.FloatText(value=1620.0, description="Cp [J/kgK]:", layout=L_(width='180px'))
    w_ca_k = widgets.FloatText(value=0.138, description="k [W/mK]:", layout=L_(width='180px'))
    w_ca_mu = widgets.FloatText(value=0.0034, description="\u03bc [Pa\u00b7s]:", layout=L_(width='180px'))
    w_cb_name = widgets.Text(value="Dowtherm B", description="Fluid B:", layout=L_(width='220px'))
    w_cb_rho = widgets.FloatText(value=1010.0, description="\u03c1 [kg/m\u00b3]:", layout=L_(width='180px'))
    w_cb_cp = widgets.FloatText(value=1900.0, description="Cp [J/kgK]:", layout=L_(width='180px'))
    w_cb_k = widgets.FloatText(value=0.115, description="k [W/mK]:", layout=L_(width='180px'))
    w_cb_mu = widgets.FloatText(value=0.0025, description="\u03bc [Pa\u00b7s]:", layout=L_(width='180px'))
    w_register_btn = widgets.Button(description="Register Custom Fluids", button_style='warning', icon='plus-circle')
    w_custom_log = widgets.Output()

    def on_register_click(b):
        with w_custom_log:
            clear_output()
            opts = list(fluids)
            for nm, rho, cp_, k_, mu_, tag in ((w_ca_name.value.strip(), w_ca_rho.value, w_ca_cp.value, w_ca_k.value, w_ca_mu.value, "A"),
                                               (w_cb_name.value.strip(), w_cb_rho.value, w_cb_cp.value, w_cb_k.value, w_cb_mu.value, "B")):
                if nm:
                    if min(rho, cp_, k_, mu_) <= 0:
                        print(f"\u274c {nm}: all properties must be > 0")
                        continue
                    CUSTOM_FLUIDS[nm] = (rho, cp_, k_, mu_)
            for nm in CUSTOM_FLUIDS:
                opts.append((f"{nm} (Custom)", nm))
            hv, cv = w_hf.value, w_cf.value
            w_hf.options = opts
            w_cf.options = opts
            w_hf.value, w_cf.value = hv, cv
            print(f"\u2713 Registered custom fluids (constant properties): {list(CUSTOM_FLUIDS.keys())}")
    w_register_btn.on_click(on_register_click)

    # ---- geometry (Design mode) -------------------------------------------------------
    w_Uo = widgets.BoundedFloatText(value=1000.0, min=10.0, max=6000.0, step=50.0, description='Uo assumed [W/m\u00b2K]:', style=desc_w, layout=L_(width='280px'))
    w_do = widgets.Dropdown(options=[('15.88 mm (5/8")', 15.88), ('19.05 mm (3/4")', 19.05), ('25.40 mm (1")', 25.40), ('31.75 mm (1 1/4")', 31.75)],
                            value=19.05, description='Tube OD:', layout=L_(width='260px'))
    w_layout = widgets.Dropdown(options=[('30\u00b0 triangular', 30), ('45\u00b0 rotated square', 45), ('90\u00b0 square', 90)], value=90, description='Layout:', layout=L_(width='260px'))
    w_passes = widgets.Dropdown(options=[1, 2, 4, 6, 8], value=2, description='Tube passes:', layout=L_(width='200px'))
    w_L = widgets.BoundedFloatText(value=4.88, min=1.0, max=12.0, step=0.1, description='Tube L [m]:', layout=L_(width='220px'))
    w_head = widgets.Dropdown(options=[('Fixed tubesheet', 'fixed'), ('U-tube', 'u_tube'), ('Split-ring floating head', 'split_ring'), ('Pull-through floating head', 'pull_through')],
                              value='fixed', description='Head type:', layout=L_(width='320px'))
    w_Bc = widgets.BoundedFloatText(value=0.25, min=0.15, max=0.45, step=0.01, description='Baffle cut Bc:', layout=L_(width='220px'))
    w_lBf = widgets.BoundedFloatText(value=0.3, min=0.1, max=1.0, step=0.05, description='lB/Ds:', layout=L_(width='200px'))
    w_Pt = widgets.BoundedFloatText(value=0.0, min=0.0, max=100.0, step=0.5, description='Pitch Pt [mm] (0=Table 3):', style=desc_w, layout=L_(width='300px'))
    w_tema = widgets.Dropdown(options=['B', 'C', 'R'], value='B', description='TEMA class:', layout=L_(width='180px'))

    design_container = widgets.VBox([
        widgets.HTML("<b>TRIAL GEOMETRY (Design mode)</b>"),
        widgets.HBox([w_Uo, w_do, w_layout, w_passes]),
        widgets.HBox([w_L, w_head, w_Bc, w_lBf]),
        widgets.HBox([w_Pt, w_tema]),
    ], layout=L_(padding='10px', border='1px solid #4CAF50', margin='5px 0px'))

    # ---- search ranges (Search mode) ---------------------------------------------------
    w_s_do = widgets.SelectMultiple(options=[15.88, 19.05, 25.4, 31.75], value=(19.05,), description='Tube OD [mm]:', style=desc_w, rows=4, layout=L_(width='170px'))
    w_s_pass = widgets.SelectMultiple(options=[1, 2, 4, 6, 8], value=(2, 4), description='Passes:', rows=5, layout=L_(width='150px'))
    w_s_lay = widgets.SelectMultiple(options=[30, 45, 90], value=(30, 90), description='Layout [\u00b0]:', rows=3, layout=L_(width='150px'))
    w_s_L = widgets.SelectMultiple(options=[3.05, 3.66, 4.88, 6.1, 7.32, 9.14], value=(3.66, 4.88, 6.1), description='L [m]:', rows=6, layout=L_(width='150px'))
    w_s_Bc = widgets.SelectMultiple(options=[0.20, 0.25, 0.30, 0.35, 0.40], value=(0.25, 0.30), description='Bc:', rows=5, layout=L_(width='150px'))
    w_s_lB = widgets.SelectMultiple(options=[0.2, 0.3, 0.4, 0.5, 0.6], value=(0.2, 0.3, 0.4), description='lB/Ds:', rows=5, layout=L_(width='150px'))
    w_s_top = widgets.IntSlider(value=10, min=3, max=25, description='Show top:', layout=L_(width='260px'))
    search_container = widgets.VBox([
        widgets.HTML("<b>STEP-9 SEARCH SPACE</b> (Ctrl/Cmd-click for multiple values; other settings come from the Design box and Advanced)"),
        widgets.HBox([w_s_do, w_s_pass, w_s_lay, w_s_L, w_s_Bc, w_s_lB]),
        w_s_top,
    ], layout=L_(padding='10px', border='1px solid #2196F3', margin='5px 0px', display='none'))

    # ---- advanced Config options --------------------------------------------------------
    w_ca = widgets.BoundedFloatText(value=0.0, min=0.0, max=20.0, step=0.5, description='Tubesheet CA [mm]:', style=desc_w, layout=L_(width='260px'))
    w_ts = widgets.BoundedFloatText(value=0.0, min=0.0, max=300.0, step=1.0, description='Tubesheet [mm] (0=TEMA min):', style=desc_w, layout=L_(width='320px'))
    w_ded = widgets.Checkbox(value=True, description='Tubesheets reduce heat-transfer length', indent=False, layout=L_(width='320px'))
    w_rnd = widgets.Checkbox(value=False, description='Round shell up to standard size', indent=False, layout=L_(width='320px'))
    w_nsh = widgets.BoundedIntText(value=0, min=0, max=6, description='Shells in series (0=auto):', style=desc_w, layout=L_(width='260px'))
    w_seal = widgets.BoundedIntText(value=0, min=0, max=10, description='Sealing-strip pairs:', style=desc_w, layout=L_(width='240px'))
    w_lanes = widgets.BoundedIntText(value=-1, min=-1, max=4, description='Pass lanes (-1=auto):', style=desc_w, layout=L_(width='240px'))
    w_lanew = widgets.BoundedFloatText(value=19.05, min=0.0, max=60.0, step=1.0, description='Lane width [mm]:', style=desc_w, layout=L_(width='240px'))
    w_Fw = widgets.Dropdown(options=[('Fig. 10 chart', 'chart'), ('Taborek Eq. 22', 'taborek')], value='chart', description='Fw method:', layout=L_(width='260px'))
    w_FL = widgets.Dropdown(options=[('Eq. 24 + Fig. 13 chart', 'chart'), ('Taborek Eq. 25', 'taborek')], value='chart', description='FL method:', layout=L_(width='280px'))
    w_crit = widgets.Dropdown(options=[('Area margin', 'area'), ('Uo_calc vs Uo_assumed', 'Uo')], value='area', description='Accept on:', layout=L_(width='260px'))
    w_mlo = widgets.BoundedFloatText(value=10.0, min=0.0, max=100.0, step=1.0, description='Margin min [%]:', style=desc_w, layout=L_(width='220px'))
    w_mhi = widgets.BoundedFloatText(value=25.0, min=0.0, max=200.0, step=1.0, description='Margin max [%]:', style=desc_w, layout=L_(width='220px'))
    w_ldlo = widgets.BoundedFloatText(value=5.0, min=1.0, max=30.0, step=0.5, description='L/Ds min:', layout=L_(width='200px'))
    w_ldhi = widgets.BoundedFloatText(value=10.0, min=1.0, max=30.0, step=0.5, description='L/Ds max:', layout=L_(width='200px'))
    w_vel = widgets.Checkbox(value=True, description='Enforce liquid tube velocity 1-2.5 m/s', indent=False, layout=L_(width='320px'))
    w_dtb = widgets.BoundedFloatText(value=0.8, min=0.0, max=5.0, step=0.1, description='Tube-baffle gap [mm]:', style=desc_w, layout=L_(width='260px'))
    w_dsb = widgets.BoundedFloatText(value=0.0, min=0.0, max=10.0, step=0.5, description='Shell-baffle gap [mm] (0=auto):', style=desc_w, layout=L_(width='320px'))
    w_nzs = widgets.BoundedFloatText(value=0.0, min=0.0, max=500.0, step=5.0, description='Shell nozzle [mm] (0=auto):', style=desc_w, layout=L_(width='300px'))
    w_nzt = widgets.BoundedFloatText(value=0.0, min=0.0, max=500.0, step=5.0, description='Tube nozzle [mm] (0=auto):', style=desc_w, layout=L_(width='300px'))
    w_kw = widgets.BoundedFloatText(value=0.0, min=0.0, max=500.0, step=1.0, description='Wall k [W/mK] (0=16 SS):', style=desc_w, layout=L_(width='280px'))

    advanced = widgets.Accordion(children=[
        widgets.VBox([widgets.HBox([w_ca, w_ts, w_nsh]), widgets.HBox([w_ded, w_rnd]), widgets.HBox([w_kw])]),
        widgets.VBox([widgets.HBox([w_Fw, w_FL, w_crit]), widgets.HBox([w_mlo, w_mhi, w_ldlo, w_ldhi]), w_vel]),
        widgets.VBox([widgets.HBox([w_seal, w_lanes, w_lanew]), widgets.HBox([w_dtb, w_dsb]), widgets.HBox([w_nzs, w_nzt])]),
    ])
    advanced.set_title(0, 'Tubesheet / shells / wall material')
    advanced.set_title(1, 'Correction-factor methods & acceptance window')
    advanced.set_title(2, 'Leakage, bypass lanes & nozzles')
    advanced.selected_index = None

    def on_mode_change(change):
        design_container.layout.display = 'block' if change['new'] == 'design' else 'none'
        search_container.layout.display = 'block' if change['new'] == 'search' else 'none'
    w_mode.observe(on_mode_change, names='value')

    w_calc_btn = widgets.Button(description="Run Thermo-Hydraulic Simulation", button_style='success', icon='gears',
                                layout=L_(width='320px', height='42px'))
    w_output = widgets.Output()

    def collect_inputs() -> dict:
        return dict(
            mode=w_mode.value,
            hot_fluid=w_hf.value, hot_P=w_hP.value, hot_Tin=w_thi.value, hot_Tout=w_tho.value, hot_m=w_mh.value, hot_Rf=w_rfh.value,
            cold_fluid=w_cf.value, cold_P=w_cP.value, cold_Tin=w_tci.value, cold_Tout=w_tco.value, cold_m=w_mc.value, cold_Rf=w_rfc.value,
            hot_in_tubes=w_alloc.value, dP_shell_kPa=w_dps.value, dP_tube_kPa=w_dpt.value,
            Uo_assumed=w_Uo.value, do_mm=w_do.value, layout=w_layout.value, passes=w_passes.value, L=w_L.value,
            head=w_head.value, Bc=w_Bc.value, lB_frac=w_lBf.value, Pt_mm=w_Pt.value, tema_class=w_tema.value,
            corrosion_allowance_mm=w_ca.value, tubesheet_mm=w_ts.value, tubesheet_deducts_length=w_ded.value,
            round_shell_to_standard=w_rnd.value, n_shells=w_nsh.value, n_seal_pairs=w_seal.value,
            n_pass_lanes=w_lanes.value, pass_lane_mm=w_lanew.value, Fw_method=w_Fw.value, FL_method=w_FL.value,
            accept_criterion=w_crit.value, margin_lo_pct=w_mlo.value, margin_hi_pct=w_mhi.value,
            LDs_lo=w_ldlo.value, LDs_hi=w_ldhi.value, liquid_velocity_check=w_vel.value,
            delta_tb_mm=w_dtb.value, delta_sb_mm=w_dsb.value, nozzle_d_shell_mm=w_nzs.value,
            nozzle_d_tube_mm=w_nzt.value, kw=w_kw.value,
            s_do=tuple(w_s_do.value) or (19.05,), s_passes=tuple(w_s_pass.value) or (2,),
            s_layouts=tuple(w_s_lay.value) or (90,), s_L=tuple(w_s_L.value) or (4.88,),
            s_Bc=tuple(w_s_Bc.value) or (0.25,), s_lB=tuple(w_s_lB.value) or (0.3,), s_top=w_s_top.value)

    def on_calc_click(b):
        with w_output:
            clear_output(wait=True)
            run_bell_delaware(collect_inputs())
    w_calc_btn.on_click(on_calc_click)

    ui = widgets.VBox([
        widgets.HTML("<h2 style='color:#1a73e8; margin-bottom:5px;'>Shell &amp; Tube (Bell-Delaware) Design Dashboard</h2>"),
        widgets.HTML("<p style='color:#555; margin-top:0px;'>Enter process conditions for any pair of fluids "
                     "(built-in, CoolProp or custom). Enter <b>0</b> for an unknown outlet temperature or flow: give flow + both "
                     "temperatures for one stream and flow <i>or</i> outlet for the other).</p>"),
        w_mode,
        widgets.HBox([
            widgets.VBox([widgets.HTML("<b style='color:#d32f2f;'>1. HOT STREAM</b>"), w_hf, w_hP, w_thi, w_tho, w_mh, w_rfh],
                         layout=L_(padding='10px', border='1px solid #ddd', margin='5px', width='320px')),
            widgets.VBox([widgets.HTML("<b style='color:#1976d2;'>2. COLD STREAM</b>"), w_cf, w_cP, w_tci, w_tco, w_mc, w_rfc],
                         layout=L_(padding='10px', border='1px solid #ddd', margin='5px', width='320px')),
            widgets.VBox([widgets.HTML("<b style='color:#e65100;'>3. CUSTOM FLUID CREATOR</b>"),
                          widgets.HTML("<sub>FLUID A:</sub>"), widgets.HBox([w_ca_name, w_ca_rho]), widgets.HBox([w_ca_cp, w_ca_k, w_ca_mu]),
                          widgets.HTML("<sub>FLUID B:</sub>"), widgets.HBox([w_cb_name, w_cb_rho]), widgets.HBox([w_cb_cp, w_cb_k, w_cb_mu]),
                          w_register_btn, w_custom_log],
                         layout=L_(padding='10px', border='1px solid #ddd', margin='5px', width='500px')),
        ]),
        widgets.HBox([w_alloc, w_dps, w_dpt]),
        design_container,
        search_container,
        widgets.HTML("<b>ADVANCED OPTIONS</b> (every Config field of the Bell-Delaware engine)"),
        advanced,
        widgets.Box([w_calc_btn], layout=L_(margin='15px 0px')),
        w_output,
    ])
    display(ui)
    return dict(ui=ui, calc_btn=w_calc_btn, output=w_output, mode=w_mode, collect=collect_inputs,
                hot_fluid=w_hf, cold_fluid=w_cf, register_btn=w_register_btn)


DASHBOARD = build_interactive_dashboard()